# Tech Challenge Fase 3 — Predição de Alfabetização

**Execução local** sobre os microdados do INEP (Avaliação da Alfabetização / AEEB) de
2023, 2024 e 2025, lidos direto dos arquivos versionados na Fase 2.

## O que este notebook faz de diferente

O notebook anterior (`01_XGBoost_Alfabetizacao`) reportava 64,98% de acurácia no modelo por
aluno e 91,16% no agregado. Auditando os dados, os dois números não se sustentam: o primeiro
é a regra `presença → alfabetizado` disfarçada de modelo, e o segundo vem de *data leakage*.

Este notebook:

1. **Audita o vazamento em código** (§2), e usa o resultado dessa auditoria para decidir o
   desenho do modelo, em vez de apenas relatá-lo.
2. **Modela apenas os alunos presentes** (§6). Aluno ausente é não-alfabetizado por definição
   do INEP — não há predição a fazer ali, e mantê-lo na base infla todas as métricas.
3. **Reconstrói as features com defasagem temporal real**: o perfil de escola e município do
   ano *t−1* (ou de *t−1* e *t−2*) prevê o resultado do ano *t* (§3).
4. **Mede o teto informacional** do problema — o melhor resultado que *qualquer* modelo
   poderia obter com estes dados na granularidade do aluno (§6.3).
5. **Entrega a meta no nível municipal** (§7), que é onde a decisão de política pública
   acontece e onde o sinal é estável.

## Desenho de validação

| Modelo | Treino | Teste |
|---|---|---|
| Aluno (condicional à presença) | presentes de 2024, contexto 2023 | presentes de 2025, contexto 2024 |
| Município | municípios 2023 → alvo 2024 | municípios 2023+2024 → alvo 2025 |

Nunca há informação do ano-alvo nas features. O limiar de decisão é escolhido em um conjunto
de validação separado — jamais no teste.

## Métricas — e por que o F1 é a escolha certa aqui

A classe positiva é `alfabetizado = 1` (e `atinge a meta = 1` no municipal), seguindo o
enunciado, que pede prever "se um aluno será considerado **alfabetizado** ou não".

**Por que não acurácia como métrica de seleção.** A acurácia conta a fração de acertos sem
distinguir de onde eles vêm. Quando uma classe é majoritária, ela premia o modelo que
simplesmente chuta essa classe: entre os alunos presentes em 2025, 66,2% são alfabetizados, então
"prever que todos serão alfabetizados" já acerta 66,2% sem conhecimento nenhum. Um modelo com
67% de acurácia parece razoável e não é. Pior: a acurácia trata todos os erros como
equivalentes, quando errar sobre uma criança que não será alfabetizada — deixando de sinalizá-la
— tem consequência diferente do erro oposto.

**Por que F1.** O F1 é a média harmônica entre *precisão* (dos que o modelo apontou como
alfabetizados, quantos realmente são) e *recall* (dos que realmente são, quantos o modelo
encontrou). O ponto está em ser **harmônica**, não aritmética: ela pune desequilíbrio entre as
duas. Um modelo que declara todo mundo alfabetizado tem recall 1,00 e precisão baixa; um modelo
excessivamente cauteloso tem precisão alta e recall baixo. A média aritmética perdoaria ambos;
a harmônica só sobe quando as duas sobem juntas. É exatamente a propriedade que queremos num
problema onde o atalho de chutar a classe majoritária está sempre disponível.

**Por que o F1 também precisa do baseline ao lado.** Esta é a ressalva que o notebook leva a
sério. Com classe positiva majoritária, "prever tudo positivo" produz um F1 *alto* — 0,796 no
modelo por aluno. Reportar "F1 = 0,79" sem essa referência é tão enganoso quanto reportar
acurácia sozinha. Por isso toda tabela deste notebook traz a coluna `F1_baseline`, e a pergunta
que fazemos nunca é "o F1 é alto?", e sim **"o F1 supera o da alternativa trivial?"**.

**Por que a acurácia ainda aparece.** Por clareza de leitura. É a métrica mais intuitiva para
quem não trabalha com ML e ajuda a ancorar a conversa com gestores. Ela é reportada ao lado,
nunca usada para escolher modelo.

**Por que AUC e Average Precision aparecem.** Porque não dependem de limiar e medem a
capacidade de **ordenar** — que, como a §7.2 demonstra, é justamente o que estes dados
sustentam. São elas que fundamentam a entrega final de priorização de risco.

> **Nota de escopo.** O enunciado (`[IAST] - Tech Challenge - Fase 3.pdf`) não estabelece
> nenhuma meta numérica de métrica. Ao contrário: afirma que *"o foco não é apenas gerar
> métricas altas, mas produzir inteligência aplicável ao contexto educacional brasileiro"*. O
> que ele exige é a pipeline completa (imputação, transformação de variáveis, tratamento de
> data leakage, pré-processamento integrado ao modelo, validação com generalização),
> interpretabilidade via Feature Importance e SHAP, e respostas às perguntas de negócio. É
> esse o padrão pelo qual este notebook se mede.

---

## Como ler este notebook

Ele foi escrito para ser **apresentado**, não só executado. Cada seção segue a mesma lógica:
primeiro a pergunta que estamos respondendo, depois o código que a responde, depois a
conclusão em texto. As conclusões nunca são afirmadas sem o número que as sustenta.

| Seção | Pergunta que responde |
|---|---|
| §1 | Que dados temos, e como eles se comportam? |
| §2 | Quais variáveis são legítimas e quais são vazamento? **Onde o notebook original errou?** |
| §3 | Como construir features que não olhem para o futuro? |
| §4 | As features preparadas têm relação com o alvo? |
| §5 | Que perfis de município existem? |
| §6 | É possível prever aluno a aluno? **Qual o teto teórico?** |
| §7 | E no nível municipal? O que o modelo pode e não pode fazer? |
| §8 | O algoritmo escolhido faz diferença? |
| §9 | Quais variáveis pesam mais? |
| §10 | Os modelos estão bem construídos? **Há overfitting?** |
| §11 | O que isso responde para o gestor público? |

Três termos que aparecem o tempo todo e vale fixar antes de começar:

- **Data leakage (vazamento)** — usar, para prever, alguma informação que só existiria depois
  do fato previsto. Produz métricas altas e inúteis.
- **Baseline** — a alternativa trivial contra a qual o modelo precisa provar seu valor. Aqui,
  quase sempre "prever a classe majoritária".
- **Overfitting** — o modelo decora o conjunto de treino em vez de aprender o padrão. Detecta-se
  comparando treino com validação (§10).

## 1. Setup e carga dos dados

In [ ]:
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, confusion_matrix,
                             classification_report, silhouette_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42
GRADE_LIMIAR = np.arange(0.05, 0.95, 0.005)

print(f"pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
def achar_data_dir() -> Path:
    """Localiza TechChallenge_2/data subindo a partir do diretorio atual."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidato = base / "TechChallenge_2" / "data"
        if candidato.is_dir():
            return candidato
    raise FileNotFoundError("TechChallenge_2/data nao encontrado a partir de " + str(Path.cwd()))


DATA_DIR = achar_data_dir()

ARQUIVOS = {
    2023: "microdados_avaliacao_da_alfabetizacao_2023.zip",
    2024: "microdados_avaliacao_da_alfabetizacao_2024.zip",
    2025: "microdados_AEEB_2025.zip",
}

# As 3 edicoes compartilham estas colunas. 2025 traz ainda as respostas item a item,
# que nao usamos: derivam do mesmo teste que define o alvo.
COLUNAS = ["SG_UF", "ID_ALUNO", "ID_ESCOLA", "TP_DEPENDENCIA", "CO_MUNICIPIO",
           "CO_CADERNO_LP", "IN_PRESENCA_LP", "IN_PREENCHIMENTO_LP",
           "VL_PESO_ALUNO_LP", "VL_PROFICIENCIA_LP", "IN_ALFABETIZADO"]


def carregar(ano: int) -> pd.DataFrame:
    with zipfile.ZipFile(DATA_DIR / ARQUIVOS[ano]).open("DADOS/TS_ALUNO.csv") as fh:
        return pd.read_csv(fh, sep=";", encoding="latin-1", usecols=COLUNAS, low_memory=False)


DADOS = {ano: carregar(ano) for ano in ARQUIVOS}

for ano, df in DADOS.items():
    print(f"{ano}: {len(df):>9,} alunos | {df.ID_ESCOLA.nunique():>6,} escolas | "
          f"{df.CO_MUNICIPIO.nunique():>5,} municipios")

### 1.1 EDA — distribuição do alvo, cobertura e nulos

In [ ]:
resumo = pd.DataFrame([{
    "ano": ano,
    "alunos": len(df),
    "taxa_alfabetizacao": df.IN_ALFABETIZADO.mean(),
    "taxa_presenca": df.IN_PRESENCA_LP.mean(),
    "taxa_alfab_entre_presentes": df.loc[df.IN_PRESENCA_LP == 1, "IN_ALFABETIZADO"].mean(),
    "proficiencia_media": df.VL_PROFICIENCIA_LP.mean(),
    "proficiencia_nula": df.VL_PROFICIENCIA_LP.isna().mean(),
} for ano, df in DADOS.items()]).set_index("ano")
display(resumo.round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
resumo[["taxa_alfabetizacao", "taxa_alfab_entre_presentes", "taxa_presenca"]].plot.bar(
    ax=axes[0], rot=0)
axes[0].set_title("Alfabetização e presença por ano")
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=7)

for ano, df in DADOS.items():
    df.VL_PROFICIENCIA_LP.dropna().plot.kde(ax=axes[1], label=str(ano))
axes[1].axvline(743, color="red", ls="--", label="corte 743")
axes[1].set_title("Distribuição da proficiência")
axes[1].set_xlabel("proficiência LP")
axes[1].legend()

cob = pd.DataFrame({
    "escolas": [len(set(DADOS[2024].ID_ESCOLA) & set(DADOS[2023].ID_ESCOLA)) / DADOS[2024].ID_ESCOLA.nunique(),
                len(set(DADOS[2025].ID_ESCOLA) & set(DADOS[2024].ID_ESCOLA)) / DADOS[2025].ID_ESCOLA.nunique()],
    "municipios": [len(set(DADOS[2024].CO_MUNICIPIO) & set(DADOS[2023].CO_MUNICIPIO)) / DADOS[2024].CO_MUNICIPIO.nunique(),
                   len(set(DADOS[2025].CO_MUNICIPIO) & set(DADOS[2024].CO_MUNICIPIO)) / DADOS[2025].CO_MUNICIPIO.nunique()],
}, index=["2023-2024", "2024-2025"])
cob.plot.bar(ax=axes[2], rot=0, ylim=(0, 1))
axes[2].set_title("Cobertura do ano anterior")
plt.tight_layout()
plt.show()

display(cob.round(4))

## 2. Auditoria de data leakage

O PDF cobra explicitamente "tratamento de data leakage". Três testes em código estabelecem o
que é sinal e o que é vazamento — e o resultado define o desenho do modelo na §6.

### 2.1 O alvo é determinístico na presença

In [ ]:
tab = pd.crosstab(DADOS[2024].IN_PRESENCA_LP, DADOS[2024].IN_ALFABETIZADO,
                  rownames=["presente"], colnames=["alfabetizado"])
display(tab)

ausentes = DADOS[2024][DADOS[2024].IN_PRESENCA_LP == 0]
print(f"Alunos ausentes em 2024: {len(ausentes):,}")
print(f"Ausentes classificados como alfabetizados: {int(ausentes.IN_ALFABETIZADO.sum())}")
print()
print("O INEP conta aluno ausente como NAO alfabetizado por definicao.")
print("'presenca' nao e uma feature preditiva: e metade da regra que define o alvo.")
print()
for ano, df in DADOS.items():
    acc = accuracy_score(df.IN_ALFABETIZADO, df.IN_PRESENCA_LP)
    print(f"  {ano}: prever 'alfabetizado = presente' acerta {acc:.2%}")
print()
print("Era dai que vinham os 64,98% do notebook anterior: o modelo 'limpo' so aprendeu isso.")
print()
print("DEFINICAO OFICIAL (INPUT_SPSS_TS_ALUNO_2024.sps, que acompanha os microdados):")
print('  IN_PRESENCA_LP  "Indicador de presenca na prova de Lingua Portuguesa (LP)."')
print('     0 = "Ausente"   1 = "Presente"')
print()
print("E uma flag binaria de comparecimento a UM EVENTO - o dia de aplicacao da prova.")
print("NAO e percentual de frequencia as aulas. Ausencia pode ser doenca, transporte,")
print("mudanca de escola ou evasao: os microdados nao distinguem.")
print()
print("Alem disso, so e observada NO DIA DA PROVA. Um alerta precoce emitido em junho")
print("nao sabe quem faltara em novembro - a variavel nem estaria disponivel.")

### 2.2 O caderno é sorteado — não carrega sinal

In [ ]:
presentes_24 = DADOS[2024][DADOS[2024].IN_PRESENCA_LP == 1]
por_caderno = (presentes_24.groupby("CO_CADERNO_LP")
               .agg(alunos=("IN_ALFABETIZADO", "size"),
                    taxa=("IN_ALFABETIZADO", "mean"),
                    proficiencia=("VL_PROFICIENCIA_LP", "mean")))
relevantes = por_caderno[por_caderno.alunos >= 1000]

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(relevantes.index.astype(str), relevantes.taxa, color="steelblue")
ax.axhline(presentes_24.IN_ALFABETIZADO.mean(), color="red", ls="--", label="média geral")
ax.set_ylim(0.5, 0.65)
ax.set_title("Taxa de alfabetização por caderno de prova (presentes, 2024)")
ax.set_xlabel("caderno")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Amplitude entre cadernos: {relevantes.taxa.max() - relevantes.taxa.min():.4f}")
print(f"Correlacao peso amostral x alvo: "
      f"{presentes_24.VL_PESO_ALUNO_LP.corr(presentes_24.IN_ALFABETIZADO):.4f}")
print()
print("Cadernos sao equivalentes (amplitude ~1,7pp) e o peso amostral e ruido.")

### 2.3 `ID_ALUNO` não rastreia o aluno entre anos

In [ ]:
pares = (DADOS[2023].set_index("ID_ALUNO")
         .join(DADOS[2024].set_index("ID_ALUNO"), lsuffix="_23", rsuffix="_24", how="inner"))

print(f"IDs presentes nos dois anos: {len(pares):,} ({len(pares) / len(DADOS[2024]):.1%} de 2024)")
print(f"  mesma escola:    {(pares.ID_ESCOLA_23 == pares.ID_ESCOLA_24).mean():.4f}")
print(f"  mesmo municipio: {(pares.CO_MUNICIPIO_23 == pares.CO_MUNICIPIO_24).mean():.4f}")
print(f"  mesma UF:        {(pares.SG_UF_23 == pares.SG_UF_24).mean():.4f}")
print(f"  corr proficiencia: {pares.VL_PROFICIENCIA_LP_23.corr(pares.VL_PROFICIENCIA_LP_24):.4f}")
print()
print("100% na mesma UF e 0,1% na mesma escola: o ID e sequencial por ano dentro da UF.")
print("Nao existe historico individual do aluno nesta base.")

### 2.4 Conclusão da auditoria — e a decisão de desenho

O `TS_ALUNO` tem 15 colunas e **nenhuma variável do aluno** — sem sexo, raça, idade, NSE,
reprovação ou frequência ao longo do ano.

| Coluna | Status |
|---|---|
| `VL_PROFICIENCIA_LP` | **vazamento** — o alvo é `proficiencia >= 743` |
| `IN_PRESENCA_LP`, `IN_PREENCHIMENTO_LP` | **parte da definição do alvo**, e indisponíveis antes da prova |
| — | *atenção: `IN_PRESENCA_LP` é comparecimento à prova, **não** frequência escolar* |
| `CO_CADERNO_LP`, `VL_PESO_ALUNO_LP` | sem sinal (§2.2) |
| `TP_SERIE` | constante (todos 2º ano) |
| `ID_ESCOLA`, `CO_MUNICIPIO`, `SG_UF`, `TP_DEPENDENCIA` | **a única informação preditiva real** |

Daí decorrem duas decisões:

**O modelo por aluno é condicional à presença.** Como `presença = 0 ⟹ alfabetizado = 0` com
certeza, não existe problema de predição para o aluno ausente: existe uma regra
administrativa. Mantê-lo na base adiciona ~12% de acertos gratuitos que inflam acurácia e F1
sem que o modelo tenha feito nada. Treinamos, validamos e testamos **apenas sobre alunos
presentes**, e o indicador oficial do INEP é reconstruído depois pela decomposição

$$P(\text{alfabetizado}) = P(\text{presente}) \times P(\text{alfabetizado} \mid \text{presente})$$

tratada na §6.4.

**Tudo que o modelo pode saber sobre um aluno é onde ele estuda.** É isso que a §3 transforma
em features, e é isso que estabelece o teto medido na §6.3.

### 2.5 Catálogo dos erros do notebook original

Consolidando o que a auditoria expôs. Cada item abaixo é um erro que aparece em muitos
projetos de ML — vale conhecê-los pelo nome.

| # | Erro | Onde aparecia | Por que é errado | Correção |
|---|---|---|---|---|
| 1 | **Feature que é a própria definição do alvo** | `presenca` com 99,85% de importância no "modelo limpo" | `presença = 0 ⟹ alfabetizado = 0` por regra do INEP. O modelo memorizava uma definição, não aprendia um padrão. Recall 100% e zero falso negativo eram o sintoma | Remover ausentes e a variável; modelar condicionalmente (§2.4) |
| 2 | **Data leakage clássico** | Modelo agregado com 91,16% de acurácia | O alvo `taxa >= 80` é calculado dos mesmos alunos que produzem `media_portugues` e `nivel_0..8`, usados como features no **mesmo ano**. O modelo lia a resposta | Features só de anos anteriores (§3) |
| 3 | **Join sem a chave temporal** | `df_hist_2023` unido por `['id_municipio','rede']`, sem `ano` | Linhas de treino (2023) recebiam indicadores do próprio 2023; as de teste (2024), defasados. Treino e teste passaram a significar coisas diferentes, e o modelo não transferiu — ganho de só 1,5pp | `assert max(anos_hist) < ano_alvo` em toda montagem (§3) |
| 4 | **Limiar otimizado no conjunto de teste** | Célula "Otimização de Threshold" | Escolher o corte que maximiza a métrica no teste não melhora o modelo: melhora só o número reportado. É vazamento de avaliação | Limiar escolhido na validação, transportado por quantil (§7) |
| 5 | **Feature constante** | `serie` entre as features | Todos os alunos são do 2º ano. Uma coluna sem variação não carrega informação | Removida |
| 6 | **Interpretação indevida da variável** | "Combate à evasão escolar (presença = feature dominante)" | `IN_PRESENCA_LP` é comparecimento **à prova**, não frequência escolar nem evasão. Doença, transporte e mudança de escola entram no mesmo `0` | Recomendação reescrita (§11) |
| 7 | **Métrica sem baseline declarado** | "64,98% de acurácia" apresentado como resultado | Sem o baseline ao lado, o leitor não sabe que a regra trivial acertava 64,84%. O modelo não agregava nada | Baseline em toda tabela deste notebook |

E um erro que cometemos **durante esta revisão**, que vale registrar pela mesma razão:

| 8 | **Comparar métricas de conjuntos de teste diferentes** | Comparar 64,98% (testado em 2024) com ~70% (testado em 2025) como se fosse melhora | A regra trivial rende 64,84% em 2024 e 70,00% em 2025, porque a taxa de ausência e a de alfabetização mudaram. Quase todo o "ganho" era troca de ano | Comparar sempre contra o baseline **do mesmo conjunto** |

## 3. Engenharia de features com defasagem temporal

A correção central em relação ao notebook anterior. Lá, o join do histórico municipal era
feito por `['id_municipio','rede']` **sem o ano** — as linhas de treino (2023) recebiam
indicadores do próprio 2023 (vazamento) enquanto as de teste (2024) recebiam defasagem real.
Treino e teste significavam coisas diferentes, e o modelo não transferia.

Aqui o perfil sempre vem de anos anteriores ao alvo, nos dois conjuntos. A função aceita
**mais de um ano de histórico**: agregar 2023+2024 para prever 2025 dobra a amostra que
estima o nível de cada unidade e reduz o ruído — ganho medido na §7.

**A ideia central desta seção.** Não temos nenhuma variável do aluno (§2.4). O que temos é a
identidade da escola e do município onde ele estuda. Sozinhos, `ID_ESCOLA` e `CO_MUNICIPIO`
são apenas códigos — um número de escola não diz nada a um modelo. O que os torna úteis é
**resumir o que aconteceu naquela escola e naquele município no passado**: qual foi a taxa de
alfabetização, qual a proficiência média, quanto as escolas variam entre si.

É esse resumo histórico que vira feature. A regra que não pode ser quebrada: o resumo precisa
vir de **anos anteriores** ao que estamos prevendo. Se um aluno de 2025 recebe o perfil da sua
escola em 2025, o modelo está vendo o resultado que deveria prever — e foi exatamente esse o
erro nº 3 do catálogo (§2.5). Por isso a função começa com um `assert`.

**Por que o perfil entre presentes (`taxa_pres`) importa.** Calculamos duas taxas para cada
unidade: a geral (que inclui ausentes contados como não-alfabetizados) e a restrita aos
presentes. A segunda mede aprendizagem; a primeira mistura aprendizagem com comparecimento.
Como o modelo da §6 é condicional à presença, é a segunda que interessa a ele.

In [ ]:
def perfis(anos_hist) -> tuple:
    """Perfil de escola e de municipio agregando uma ou mais edicoes ANTERIORES ao alvo.

    A taxa entre presentes (`*_taxa_pres`) e a que interessa ao modelo condicional da §6:
    ela separa o efeito de aprendizagem do efeito de ausencia.
    """
    d = pd.concat([DADOS[a] for a in anos_hist])
    pres = d[d.IN_PRESENCA_LP == 1]

    esc = d.groupby("ID_ESCOLA").agg(
        taxa=("IN_ALFABETIZADO", "mean"), n=("IN_ALFABETIZADO", "size"),
        prof=("VL_PROFICIENCIA_LP", "mean"), prof_std=("VL_PROFICIENCIA_LP", "std"),
        p25=("VL_PROFICIENCIA_LP", lambda s: s.quantile(0.25)),
        p75=("VL_PROFICIENCIA_LP", lambda s: s.quantile(0.75)),
        presenca=("IN_PRESENCA_LP", "mean"))
    esc["taxa_pres"] = pres.groupby("ID_ESCOLA").IN_ALFABETIZADO.mean()

    mun = d.groupby("CO_MUNICIPIO").agg(
        taxa=("IN_ALFABETIZADO", "mean"), n=("IN_ALFABETIZADO", "size"),
        prof=("VL_PROFICIENCIA_LP", "mean"), prof_std=("VL_PROFICIENCIA_LP", "std"),
        p10=("VL_PROFICIENCIA_LP", lambda s: s.quantile(0.10)),
        p25=("VL_PROFICIENCIA_LP", lambda s: s.quantile(0.25)),
        p75=("VL_PROFICIENCIA_LP", lambda s: s.quantile(0.75)),
        p90=("VL_PROFICIENCIA_LP", lambda s: s.quantile(0.90)),
        presenca=("IN_PRESENCA_LP", "mean"), n_esc=("ID_ESCOLA", "nunique"))
    mun["taxa_pres"] = pres.groupby("CO_MUNICIPIO").IN_ALFABETIZADO.mean()

    # desigualdade interna do municipio: como suas escolas se distribuem
    por_escola = d.groupby(["CO_MUNICIPIO", "ID_ESCOLA"]).IN_ALFABETIZADO.mean().groupby("CO_MUNICIPIO")
    mun["esc_spread"] = por_escola.std()
    mun["esc_min"] = por_escola.min()
    mun["esc_max"] = por_escola.max()

    esc.columns = [f"esc_{c}" for c in esc.columns]
    mun.columns = [f"mun_{c}" for c in mun.columns]
    return esc, mun


PERFIL_1ANO = {ano: perfis([ano]) for ano in DADOS}

print("Perfis por ano:")
for ano in DADOS:
    e, m = PERFIL_1ANO[ano]
    print(f"  {ano}: {len(e):>6,} escolas | {len(m):>5,} municipios")
print()
print("Colunas do perfil municipal:", list(PERFIL_1ANO[2023][1].columns))

In [ ]:
def montar_aluno(ano_alvo: int, anos_hist, so_presentes: bool = True) -> pd.DataFrame:
    """Alunos do ano-alvo enriquecidos com o contexto de anos anteriores.

    Nenhuma coluna derivada do resultado do ano-alvo entra nas features - so o rotulo.
    """
    assert max(anos_hist) < ano_alvo, "o contexto tem que vir de anos anteriores ao alvo"

    esc, mun = perfis(anos_hist)
    base = DADOS[ano_alvo]
    if so_presentes:
        base = base[base.IN_PRESENCA_LP == 1]

    X = base.join(esc, on="ID_ESCOLA").join(mun, on="CO_MUNICIPIO").copy()
    X["esc_gap"] = X.esc_taxa_pres - X.mun_taxa_pres   # posicao da escola no seu municipio
    X["esc_cob"] = X.esc_taxa.notna().astype(int)      # escola sem historico e informacao
    X["ano_alvo"] = ano_alvo
    X["anos_contexto"] = str(sorted(anos_hist))
    return X


# Modelo condicional: so alunos presentes (ver §2.4)
ALUNO_TREINO = montar_aluno(2024, [2023])
ALUNO_TESTE = montar_aluno(2025, [2024])

print(f"treino: {len(ALUNO_TREINO):,} alunos PRESENTES de 2024 (contexto 2023)")
print(f"teste : {len(ALUNO_TESTE):,} alunos PRESENTES de 2025 (contexto 2024)")
print()
print(f"taxa de alfabetizados no treino: {ALUNO_TREINO.IN_ALFABETIZADO.mean():.4f}")
print(f"taxa de alfabetizados no teste : {ALUNO_TESTE.IN_ALFABETIZADO.mean():.4f}")
print(f"cobertura de escola no teste   : {ALUNO_TESTE.esc_cob.mean():.1%}")

## 4. Correlação entre as features preparadas

In [ ]:
FEAT_NUM_ALUNO = [
    "esc_taxa", "esc_n", "esc_prof", "esc_prof_std", "esc_p25", "esc_p75",
    "esc_presenca", "esc_taxa_pres", "esc_gap", "esc_cob",
    "mun_taxa", "mun_n", "mun_prof", "mun_prof_std", "mun_p10", "mun_p25", "mun_p75",
    "mun_p90", "mun_presenca", "mun_taxa_pres", "mun_n_esc",
    "mun_esc_spread", "mun_esc_min", "mun_esc_max",
]
FEAT_CAT_ALUNO = ["TP_DEPENDENCIA", "SG_UF"]

amostra = ALUNO_TREINO.sample(250_000, random_state=RANDOM_STATE)
corr = amostra[FEAT_NUM_ALUNO + ["IN_ALFABETIZADO"]].corr()

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 6.5},
            linewidths=0.4, cbar_kws={"shrink": 0.6}, ax=ax)
ax.set_title("Correlação entre features preparadas e o alvo\n(250k alunos presentes de 2024)")
plt.tight_layout()
plt.show()

alvo = corr["IN_ALFABETIZADO"].drop("IN_ALFABETIZADO").sort_values(key=abs, ascending=False)
display(alvo.to_frame("corr_com_alvo").round(4))

**Leitura do heatmap.** Sem `presenca` e `preenchimento` na base — removidos por serem parte
da definição do alvo — as correlações com o alvo caem para a faixa de 0,05 a 0,15. Isso é
honesto: é a magnitude real do sinal disponível.

As features de contexto correlacionam fortemente **entre si** (`mun_prof` com `mun_p25`,
`mun_p75`, `mun_taxa`…), porque são recortes da mesma distribuição de proficiência. Justifica
usar regularização (`reg_lambda`) e modelos de árvore, que toleram colinearidade, em vez de
tentar selecionar features "independentes".

## 5. Clusterização de municípios como enriquecimento

Agrupar municípios por perfil educacional responde à pergunta de negócio *"quais regiões
possuem padrões semelhantes?"* e gera uma feature categórica para os modelos.

O KMeans é ajustado **apenas no ano de contexto do treino (2023)** e depois aplicado aos
demais anos — clusterizar com todos os anos juntos vazaria informação do futuro.

In [ ]:
FEAT_CLUSTER = ["mun_taxa_pres", "mun_prof", "mun_prof_std", "mun_presenca", "mun_esc_spread"]

base_cluster = PERFIL_1ANO[2023][1][FEAT_CLUSTER].dropna()
scaler_cluster = StandardScaler().fit(base_cluster)
Z = scaler_cluster.transform(base_cluster)

ks = range(2, 11)
inercias, silhuetas = [], []
n_sil = min(5000, len(Z))
amostra_sil = np.random.RandomState(RANDOM_STATE).choice(len(Z), n_sil, replace=False)
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Z)
    inercias.append(km.inertia_)
    silhuetas.append(silhouette_score(Z[amostra_sil], km.labels_[amostra_sil]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(ks), inercias, "o-")
axes[0].set_title("Cotovelo")
axes[0].set_xlabel("k")
axes[0].set_ylabel("inércia")
axes[1].plot(list(ks), silhuetas, "o-", color="darkorange")
axes[1].set_title("Silhueta")
axes[1].set_xlabel("k")
plt.tight_layout()
plt.show()

K = 6
print(f"k escolhido = {K}: o cotovelo estabiliza e a silhueta ainda e razoavel.")

In [ ]:
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(Z)


def rotular_clusters(perfil_mun: pd.DataFrame) -> pd.Series:
    """Aplica o KMeans treinado em 2023 ao perfil de qualquer ano."""
    ok = perfil_mun[FEAT_CLUSTER].dropna()
    return pd.Series(kmeans.predict(scaler_cluster.transform(ok)),
                     index=ok.index, name="mun_cluster")


CLUSTERS = {ano: rotular_clusters(PERFIL_1ANO[ano][1]) for ano in DADOS}

# o cluster vem do ano de CONTEXTO, igual as demais features
ALUNO_TREINO["mun_cluster"] = ALUNO_TREINO.CO_MUNICIPIO.map(CLUSTERS[2023])
ALUNO_TESTE["mun_cluster"] = ALUNO_TESTE.CO_MUNICIPIO.map(CLUSTERS[2024])

perfil_clusters = (PERFIL_1ANO[2023][1].join(CLUSTERS[2023])
                   .groupby("mun_cluster")
                   .agg(municipios=("mun_taxa", "size"),
                        taxa_presentes=("mun_taxa_pres", "mean"),
                        proficiencia=("mun_prof", "mean"),
                        presenca=("mun_presenca", "mean"),
                        desigualdade_interna=("mun_esc_spread", "mean"),
                        escolas_por_mun=("mun_n_esc", "mean"))
                   .sort_values("taxa_presentes"))
display(perfil_clusters.round(3))

pm = PERFIL_1ANO[2023][1].join(CLUSTERS[2023]).dropna(subset=["mun_cluster"])
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.scatterplot(data=pm, x="mun_prof", y="mun_taxa_pres", hue="mun_cluster",
                palette="viridis", s=10, alpha=0.6, ax=axes[0], legend="full")
axes[0].set_title("Municípios por proficiência x taxa (contexto 2023)")
sns.boxplot(data=pm, x="mun_cluster", y="mun_esc_spread", palette="viridis", ax=axes[1])
axes[1].set_title("Desigualdade entre escolas dentro do município")
plt.tight_layout()
plt.show()

## 6. Modelo por aluno — condicional à presença

Treino, validação e teste contêm **apenas alunos presentes**, pela razão da §2.4. O baseline
correspondente é "prever que todo aluno presente será alfabetizado", que em 2025 acerta
66,15% e tem recall 1,0000.

Pipeline scikit-learn com imputação, padronização e one-hot integrados ao classificador — o
pré-processamento é ajustado **dentro** do `fit`, então nada do teste influencia a
transformação.

In [ ]:
FEATURES_ALUNO = FEAT_NUM_ALUNO + FEAT_CAT_ALUNO + ["mun_cluster"]
CAT_ALUNO = FEAT_CAT_ALUNO + ["mun_cluster"]

# Guarda anti-leakage
PROIBIDAS = {"VL_PROFICIENCIA_LP", "IN_ALFABETIZADO", "ID_ALUNO", "ID_ESCOLA",
             "CO_MUNICIPIO", "IN_PRESENCA_LP", "IN_PREENCHIMENTO_LP", "VL_PESO_ALUNO_LP"}
assert not (set(FEATURES_ALUNO) & PROIBIDAS), "feature proibida no conjunto"
assert ALUNO_TREINO.IN_PRESENCA_LP.eq(1).all(), "treino deve conter so presentes"
assert ALUNO_TESTE.IN_PRESENCA_LP.eq(1).all(), "teste deve conter so presentes"
print("Guarda anti-leakage: OK")
print(f"{len(FEATURES_ALUNO)} features ({len(FEAT_NUM_ALUNO)} numericas, {len(CAT_ALUNO)} categoricas)")
print("Note que presenca/preenchimento NAO estao entre elas.")


def construir_pipeline(modelo, num, cat):
    pre = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), num),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
    ])
    return Pipeline([("preprocessor", pre), ("modelo", modelo)])


def limiar_por_f1(y_val, p_val) -> float:
    """Limiar que maximiza F1 na VALIDACAO (nunca no teste)."""
    return float(GRADE_LIMIAR[int(np.argmax(
        [f1_score(y_val, (p_val >= t).astype(int)) for t in GRADE_LIMIAR]))])


def avaliar(y, proba, limiar, nome=""):
    pred = (proba >= limiar).astype(int)
    return {
        "modelo": nome, "limiar": limiar,
        "F1": f1_score(y, pred), "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred), "Accuracy": accuracy_score(y, pred),
        "AUC": roc_auc_score(y, proba),
        "AP_risco": average_precision_score(1 - y, 1 - proba),
    }

In [ ]:
X_full, y_full = ALUNO_TREINO[FEATURES_ALUNO], ALUNO_TREINO.IN_ALFABETIZADO
X_teste, y_teste = ALUNO_TESTE[FEATURES_ALUNO], ALUNO_TESTE.IN_ALFABETIZADO

X_tr, X_val, y_tr, y_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y_full)

modelo_aluno = construir_pipeline(
    XGBClassifier(n_estimators=400, learning_rate=0.06, max_depth=6,
                  min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
                  reg_alpha=0.1, reg_lambda=2.0, tree_method="hist",
                  n_jobs=-1, eval_metric="logloss", random_state=RANDOM_STATE),
    FEAT_NUM_ALUNO, CAT_ALUNO)

modelo_aluno.fit(X_tr, y_tr)
p_val = modelo_aluno.predict_proba(X_val)[:, 1]
LIMIAR = limiar_por_f1(y_val, p_val)
p_teste = modelo_aluno.predict_proba(X_teste)[:, 1]

print(f"Treinado em {len(X_tr):,} alunos presentes de 2024.")
print(f"Limiar (maximiza F1 na validacao): {LIMIAR:.3f}")

# baseline: prever que todo presente sera alfabetizado
base_pred = np.ones(len(y_teste), dtype=int)
linhas = [
    {"modelo": "baseline: todos alfabetizado", "limiar": np.nan,
     "F1": f1_score(y_teste, base_pred), "Precision": precision_score(y_teste, base_pred),
     "Recall": 1.0, "Accuracy": accuracy_score(y_teste, base_pred),
     "AUC": np.nan, "AP_risco": np.nan},
    avaliar(y_teste, p_teste, LIMIAR, "XGBoost"),
]
resultado_aluno = pd.DataFrame(linhas).set_index("modelo")
display(resultado_aluno.round(4))

metricas_aluno = linhas[1]
print(classification_report(y_teste, (p_teste >= LIMIAR).astype(int),
                            target_names=["Nao alfabetizado", "Alfabetizado"]))
print(f"AP na classe em risco: {metricas_aluno['AP_risco']:.4f} "
      f"(acaso = {1 - y_teste.mean():.4f})")

In [ ]:
cm = confusion_matrix(y_teste, (p_teste >= LIMIAR).astype(int))
fpr, tpr, _ = roc_curve(y_teste, p_teste)
prec_c, rec_c, _ = precision_recall_curve(1 - y_teste, 1 - p_teste)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=axes[0],
            xticklabels=["Nao alf.", "Alf."], yticklabels=["Nao alf.", "Alf."])
axes[0].set_title(f"Matriz de confusão (F1={metricas_aluno['F1']:.4f})")
axes[0].set_xlabel("previsto")
axes[0].set_ylabel("real")

axes[1].plot(fpr, tpr, label=f"AUC = {metricas_aluno['AUC']:.4f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("Curva ROC (só presentes)")
axes[1].set_xlabel("FPR")
axes[1].set_ylabel("TPR")
axes[1].legend()

axes[2].plot(rec_c, prec_c, color="darkgreen")
axes[2].axhline(1 - y_teste.mean(), color="red", ls="--", label="acaso")
axes[2].set_title(f"PR da classe EM RISCO (AP={metricas_aluno['AP_risco']:.4f})")
axes[2].set_xlabel("recall")
axes[2].set_ylabel("precision")
axes[2].legend()
plt.tight_layout()
plt.show()

### 6.1 Por que o F1 mal se move — a aritmética do baseline

O baseline "todos alfabetizado" tem **recall exatamente 1,0000**: não produz nenhum falso
negativo. Para o modelo ganhar F1 ele precisa marcar alunos como não-alfabetizados, e cada
acerto vem acompanhado de erros.

Escrevendo $N_1$ = alfabetizados e $N_0$ = não-alfabetizados presentes, o baseline tem
$F_1 = 2N_1 / (2N_1 + N_0)$. Marcando $a$ alunos corretamente e $b$ incorretamente como
classe 0, o F1 só aumenta se

$$\frac{a}{b} > \frac{N_1 + N_0}{N_1}$$

A célula abaixo calcula esse limiar e verifica se existe algum subconjunto de alunos que o
satisfaça.

In [ ]:
N1 = int(y_teste.sum())
N0 = int((y_teste == 0).sum())
razao_necessaria = (N1 + N0) / N1
precisao_necessaria = (N1 + N0) / (2 * N1 + N0)

print(f"alfabetizados N1 = {N1:,} | nao-alfabetizados N0 = {N0:,}")
print(f"F1 do baseline = {2 * N1 / (2 * N1 + N0):.4f}")
print(f"razao a/b necessaria = {razao_necessaria:.3f}")
print(f"-> precisao minima ao apontar a classe 0: {precisao_necessaria:.1%}")
print(f"   (taxa-base de nao-alfabetizados: {1 - y_teste.mean():.1%})")
print()

ordem = np.argsort(p_teste)
neg_ordenado = (y_teste.values[ordem] == 0).astype(int)
precisao_acum = np.cumsum(neg_ordenado) / np.arange(1, len(ordem) + 1)

fig, ax = plt.subplots(figsize=(9, 4))
ks = np.arange(1, len(ordem) + 1)
ax.plot(ks, precisao_acum, lw=1)
ax.axhline(precisao_necessaria, color="red", ls="--",
           label=f"precisão necessária ({precisao_necessaria:.1%})")
ax.axhline(1 - y_teste.mean(), color="gray", ls=":", label="taxa-base")
ax.set_xscale("log")
ax.set_xlabel("k alunos de menor probabilidade marcados como classe 0 (escala log)")
ax.set_ylabel("precisão acumulada")
ax.set_title("Só um subconjunto muito pequeno atinge a precisão necessária")
ax.legend()
plt.tight_layout()
plt.show()

viaveis = int((precisao_acum > precisao_necessaria).sum())
print(f"alunos em subconjuntos viaveis: {viaveis:,} "
      f"({viaveis / len(ordem):.2%} dos presentes)")
print()
melhor_f1 = max((f1_score(y_teste, (p_teste >= t).astype(int)), t) for t in GRADE_LIMIAR)
print(f"teto de F1 deste modelo (limiar varrido no proprio teste): {melhor_f1[0]:.4f}")
print(f"F1 do baseline                                           : {2*N1/(2*N1+N0):.4f}")
print()
print("O modelo empata com o baseline no F1. O ganho possivel existe, mas cabe")
print("num subconjunto pequeno demais para mover a metrica.")

### 6.2 Quanto cada bloco de feature contribui

In [ ]:
blocos = {
    "só contexto do município": [c for c in FEAT_NUM_ALUNO if c.startswith("mun_")],
    "só contexto da escola": [c for c in FEAT_NUM_ALUNO if c.startswith("esc_")],
    "escola + município": FEAT_NUM_ALUNO,
    "+ UF/rede/cluster (completo)": FEATURES_ALUNO,
}

linhas = []
for nome, feats in blocos.items():
    num = [c for c in feats if c in FEAT_NUM_ALUNO]
    cat = [c for c in feats if c in CAT_ALUNO]
    mdl = construir_pipeline(
        XGBClassifier(n_estimators=300, learning_rate=0.08, max_depth=6, subsample=0.8,
                      colsample_bytree=0.8, min_child_weight=20, reg_lambda=2.0,
                      tree_method="hist", n_jobs=-1, eval_metric="logloss",
                      random_state=RANDOM_STATE), num, cat)
    mdl.fit(X_tr[feats], y_tr)
    lim = limiar_por_f1(y_val, mdl.predict_proba(X_val[feats])[:, 1])
    pt = mdl.predict_proba(X_teste[feats])[:, 1]
    r = avaliar(y_teste, pt, lim, nome)
    r["n_features"] = len(feats)
    linhas.append(r)

ablacao = pd.DataFrame(linhas).set_index("modelo")[["n_features", "F1", "Accuracy", "AUC", "AP_risco"]]
display(ablacao.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
ablacao[["F1", "Accuracy", "AUC", "AP_risco"]].plot.bar(ax=ax, rot=15)
ax.axhline(2 * N1 / (2 * N1 + N0), color="red", ls="--", lw=1, label="F1 do baseline")
ax.set_ylim(0.3, 0.9)
ax.set_title("Contribuição incremental dos blocos de feature (teste 2025, só presentes)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 6.3 O teto informacional — qual o melhor resultado possível com estes dados

Esta é a seção mais importante do notebook. Ela responde a uma pergunta que normalmente não
se faz: **qual é o melhor resultado possível para este problema, com estes dados?**

O raciocínio é o seguinte. Toda informação disponível sobre um aluno é do nível da escola
(§2.4) — dois colegas da mesma sala são, para o modelo, idênticos. Ele obrigatoriamente dará a
mesma previsão para os dois. Logo, o melhor que qualquer modelo pode fazer é, para cada
escola, acertar a classe majoritária dela.

Imagine então um **oráculo**: um modelo mágico que já soubesse a taxa verdadeira de alfabetização
de cada escola no próprio ano que queremos prever, e que previsse a classe majoritária de cada
uma. Esse oráculo é impossível de construir — ele usa o futuro. Mas o desempenho dele é um
**limite superior matemático**: nenhum modelo baseado em features de escola pode superá-lo,
porque nenhum modelo consegue distinguir dois alunos da mesma escola.

Saber esse teto muda a conversa. Sem ele, um resultado de 66% parece fracasso de modelagem e
convida a tentar mais algoritmos indefinidamente. Com ele, sabe-se se ainda há espaço a ganhar
ou se o limite já foi alcançado — e qualquer expectativa acima do teto pode ser descartada por
aritmética, não por opinião. É o que a célula seguinte calcula.

In [ ]:
def teto_informacional(ano: int) -> dict:
    """Teto entre presentes, e o teto equivalente no indicador oficial (com ausentes)."""
    df = DADOS[ano]
    frac_ausente = (df.IN_PRESENCA_LP == 0).mean()
    pres = df[df.IN_PRESENCA_LP == 1]
    q = pres.groupby("ID_ESCOLA").IN_ALFABETIZADO.transform("mean")
    acc_pres = float(np.mean(np.maximum(q, 1 - q)))
    base = pres.IN_ALFABETIZADO.mean()
    return {"ano": ano, "ausentes": frac_ausente,
            "baseline_presentes": max(base, 1 - base),
            "TETO_presentes": acc_pres,
            "TETO_indicador_oficial": frac_ausente + (1 - frac_ausente) * acc_pres}


tetos = pd.DataFrame([teto_informacional(a) for a in (2024, 2025)]).set_index("ano")
display(tetos.round(4))

teto_pres = tetos.loc[2025, "TETO_presentes"]
base_pres = tetos.loc[2025, "baseline_presentes"]
teto_oficial = tetos.loc[2025, "TETO_indicador_oficial"]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
itens = {
    "baseline\n(todos alf.)": base_pres,
    "este\nmodelo": metricas_aluno["Accuracy"],
    "TETO\n(oráculo)": teto_pres,
}
axes[0].bar(list(itens), list(itens.values()),
            color=["lightgray", "steelblue", "seagreen"])
for i, v in enumerate(itens.values()):
    axes[0].text(i, v + 0.003, f"{v:.2%}", ha="center", fontsize=9)
axes[0].set_ylim(0.6, 0.75)
axes[0].set_ylabel("acurácia")
axes[0].set_title("Entre alunos PRESENTES (o problema real)")

itens2 = {"este modelo\n(recomposto)":
          (1 - tetos.loc[2025, "ausentes"]) * metricas_aluno["Accuracy"] + tetos.loc[2025, "ausentes"],
          "TETO\n(oráculo)": teto_oficial}
axes[1].bar(list(itens2), list(itens2.values()), color=["steelblue", "seagreen"])
for i, v in enumerate(itens2.values()):
    axes[1].text(i, v + 0.003, f"{v:.2%}", ha="center", fontsize=9)
axes[1].set_ylim(0.6, 0.8)
axes[1].set_title("No indicador oficial (recompondo os ausentes)")
plt.tight_layout()
plt.show()

print(f"Teto entre presentes (2025):       {teto_pres:.2%}")
print(f"Teto no indicador oficial (2025):  {teto_oficial:.2%}")
print(f"Baseline entre presentes:          {base_pres:.2%}")
print(f"Espaco total disponivel:           {(teto_pres - base_pres) * 100:.2f} pp")
print()
print("Ou seja: entre o baseline trivial e o melhor modelo concebivel existem pouco mais")
print("de 4 pontos percentuais. Todo o esforco de modelagem por aluno disputa essa faixa.")
print()
print("Nenhuma engenharia de features, clusterizacao ou algoritmo novo ultrapassa esse teto:")
print("features de escola nao discriminam DENTRO da escola, e e la que mora a variacao")
print("restante (84,7% dela, como a §6.1 mostrou).")
print()
print("A conclusao nao e que o problema e insoluvel - e que ele foi formulado na")
print("granularidade errada. A §7 mostra a granularidade certa.")

### 6.4 Recompondo o indicador oficial

O indicador do INEP inclui os ausentes. Ele é reconstruído pela decomposição

$$P(\text{alfabetizado}) = P(\text{presente}) \times P(\text{alfabetizado} \mid \text{presente})$$

O segundo fator é o modelo da §6. O primeiro é um problema **separado e de interesse próprio**
— prever evasão —, treinado com as mesmas features de contexto.

In [ ]:
# Modelo de presenca: mesmo desenho, alvo = compareceu a avaliacao
pres_treino = montar_aluno(2024, [2023], so_presentes=False)
pres_teste = montar_aluno(2025, [2024], so_presentes=False)
pres_treino["mun_cluster"] = pres_treino.CO_MUNICIPIO.map(CLUSTERS[2023])
pres_teste["mun_cluster"] = pres_teste.CO_MUNICIPIO.map(CLUSTERS[2024])

Xp_tr, Xp_val, yp_tr, yp_val = train_test_split(
    pres_treino[FEATURES_ALUNO], pres_treino.IN_PRESENCA_LP,
    test_size=0.2, random_state=RANDOM_STATE, stratify=pres_treino.IN_PRESENCA_LP)

modelo_presenca = construir_pipeline(
    XGBClassifier(n_estimators=300, learning_rate=0.08, max_depth=6, min_child_weight=20,
                  subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, tree_method="hist",
                  n_jobs=-1, eval_metric="logloss", random_state=RANDOM_STATE),
    FEAT_NUM_ALUNO, CAT_ALUNO).fit(Xp_tr, yp_tr)

p_presenca = modelo_presenca.predict_proba(pres_teste[FEATURES_ALUNO])[:, 1]
auc_presenca = roc_auc_score(pres_teste.IN_PRESENCA_LP, p_presenca)
print(f"Modelo de PRESENCA (evasao): AUC = {auc_presenca:.4f}")
print(f"  taxa de presenca real em 2025: {pres_teste.IN_PRESENCA_LP.mean():.4f}")
print()

# Indicador oficial recomposto
p_alfab_dado_presente = modelo_aluno.predict_proba(pres_teste[FEATURES_ALUNO])[:, 1]
p_oficial = p_presenca * p_alfab_dado_presente
y_oficial = pres_teste.IN_ALFABETIZADO.values

lim_of = float(GRADE_LIMIAR[int(np.argmax(
    [f1_score(y_oficial, (p_oficial >= t).astype(int)) for t in GRADE_LIMIAR]))])
pred_of = (p_oficial >= lim_of).astype(int)
print("INDICADOR OFICIAL recomposto (inclui ausentes), teste 2025:")
print(f"  F1={f1_score(y_oficial, pred_of):.4f}  "
      f"Acc={accuracy_score(y_oficial, pred_of):.4f}  "
      f"AUC={roc_auc_score(y_oficial, p_oficial):.4f}")
print(f"  regra trivial 'alfabetizado = presente': "
      f"F1={f1_score(y_oficial, pres_teste.IN_PRESENCA_LP):.4f}  "
      f"Acc={accuracy_score(y_oficial, pres_teste.IN_PRESENCA_LP):.4f}")
print()
print("O AUC sobe ao incluir ausentes, mas isso NAO e habilidade preditiva nova:")
print("e o modelo separando ausente de presente, o que e definicao, nao predicao.")
print("Por isso as metricas do modelo sao reportadas entre presentes (§6).")

**O que a recomposição revela — e ela não favorece o modelo.**

A presença é quase imprevisível a partir do contexto da escola: AUC ≈ 0,60, pouco acima do
acaso. Faltar à prova depende de circunstâncias do dia e da família, não do histórico da
escola. Como consequência, o indicador oficial reconstruído fica **pior** que a regra trivial
`alfabetizado = presente` (F1 ≈ 0,74 contra 0,80).

Isso não é um defeito da decomposição: é a constatação de que, no indicador oficial, o termo
dominante é a presença, e a presença só é conhecida no dia da prova. Quem quiser prever o
indicador oficial *antes* da avaliação esbarra nisso. A decomposição está aqui como
diagnóstico honesto, não como entregável — o entregável é o modelo condicional da §6 e,
sobretudo, o municipal da §7.

## 7. Modelo municipal — a decisão que a política pública realmente toma

Gestores não intervêm aluno a aluno: alocam recursos por rede e por município. Nessa
granularidade o ruído individual se cancela e o sinal histórico é estável — a correlação da
taxa municipal entre anos consecutivos é ≈ 0,71, contra ≈ 0,16 por escola.

**Por que agregar resolve o que o modelo por aluno não conseguia.** O resultado de um aluno
individual é imprevisível: 84,7% da variação está dentro da própria escola, entre colegas
sobre os quais não há dado. Mas a *média* de ~355 alunos de um município é outra coisa — essa
variação individual se cancela ao ser somada, e o que sobra é justamente o efeito do lugar,
que é o que o passado consegue prever. Não mudamos a informação disponível; mudamos o alvo
para uma quantidade em que o ruído se anula.

Duas melhorias medidas entram aqui:

- **Histórico de dois anos** (2023+2024 para prever 2025): dobra a amostra que estima o nível
  do município. Ganho de ~2pp de AUC, estável ao longo de sementes.
- **Limiar por quantil**: em vez de transportar o valor absoluto de probabilidade da
  validação para o teste, transporta-se a *fração de municípios sinalizados*. Como a taxa de
  positivos muda entre anos (13,8% → 21,6%), o limiar absoluto não transfere. Ganho de ~8pp
  de F1, com metade da variância.

  *Exemplo concreto:* se na validação o melhor F1 saiu sinalizando os 23% municípios de maior
  probabilidade, aplicamos ao teste o corte que também sinaliza 23% — não o valor numérico
  0,17 que gerou esses 23% na validação. A posição relativa transfere entre anos; o valor
  absoluto não.

Dois alvos complementares: **`meta_80`** (o município atinge 80%?) e **`meta_inep`** (atinge a
meta que o INEP definiu para ele, que varia de 14% a 80% conforme o ponto de partida).

In [ ]:
metas = pd.read_csv(DATA_DIR / "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv.gz",
                    low_memory=False)
metas = metas[metas.ano == 2024].drop_duplicates("id_municipio").set_index("id_municipio")
print(f"Metas municipais: {len(metas):,} municipios (rede {metas.rede.unique()})")


def montar_municipio(anos_hist, ano_alvo: int, coluna_meta: str) -> pd.DataFrame:
    """Perfil municipal de anos anteriores -> o municipio atinge a meta no ano-alvo?"""
    assert max(anos_hist) < ano_alvo

    _, mun = perfis(anos_hist)
    X = mun.copy()
    X["mun_cluster"] = CLUSTERS[max(anos_hist)]
    X["uf"] = pd.concat([DADOS[a] for a in anos_hist]).groupby("CO_MUNICIPIO").SG_UF.first()
    X["meta"] = metas[coluna_meta].reindex(X.index) / 100.0
    X["gap_meta"] = X.meta - X.mun_taxa
    X["taxa_alvo"] = perfis([ano_alvo])[1].mun_taxa.reindex(X.index)

    X = X.dropna(subset=["taxa_alvo", "meta"])
    X["meta_80"] = (X.taxa_alvo >= 0.80).astype(int)
    X["meta_inep"] = (X.taxa_alvo >= X.meta).astype(int)
    return X


MUN_TREINO = montar_municipio([2023], 2024, "meta_alfabetizacao_2024")
MUN_TESTE = montar_municipio([2023, 2024], 2025, "meta_alfabetizacao_2025")

print(f"\ntreino: {len(MUN_TREINO):,} municipios (perfil 2023 -> resultado 2024)")
print(f"teste : {len(MUN_TESTE):,} municipios (perfil 2023+2024 -> resultado 2025)")
for alvo in ("meta_80", "meta_inep"):
    print(f"  {alvo}: positivos treino {MUN_TREINO[alvo].mean():.1%} | "
          f"teste {MUN_TESTE[alvo].mean():.1%}")

In [ ]:
FEAT_NUM_MUN = [c for c in MUN_TREINO.columns
                if c.startswith("mun_") and c != "mun_cluster"] + ["meta", "gap_meta"]
FEAT_CAT_MUN = ["uf", "mun_cluster"]
FEATURES_MUN = FEAT_NUM_MUN + FEAT_CAT_MUN

assert "taxa_alvo" not in FEATURES_MUN and "meta_80" not in FEATURES_MUN
print(f"{len(FEATURES_MUN)} features municipais, todas dos anos de contexto")

PARAMS_MUN = dict(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=5, reg_lambda=2.0,
                  tree_method="hist", n_jobs=-1, eval_metric="logloss",
                  random_state=RANDOM_STATE)


def treinar_municipal(alvo: str, verbose: bool = True):
    """Treina, escolhe a fracao a sinalizar na validacao e aplica por QUANTIL no teste."""
    Xtr, ytr = MUN_TREINO[FEATURES_MUN], MUN_TREINO[alvo]
    Xte, yte = MUN_TESTE[FEATURES_MUN], MUN_TESTE[alvo]

    Xa, Xv, ya, yv = train_test_split(Xtr, ytr, test_size=0.25,
                                      random_state=RANDOM_STATE, stratify=ytr)
    mdl = construir_pipeline(XGBClassifier(**PARAMS_MUN), FEAT_NUM_MUN, FEAT_CAT_MUN)
    mdl.fit(Xa, ya)
    pv = mdl.predict_proba(Xv)[:, 1]
    lim_abs = limiar_por_f1(yv, pv)
    frac_sinalizada = float((pv >= lim_abs).mean())

    mdl.fit(Xtr, ytr)
    pt = mdl.predict_proba(Xte)[:, 1]
    lim_quantil = float(np.quantile(pt, 1 - frac_sinalizada))

    r_abs = avaliar(yte, pt, lim_abs, f"{alvo} (limiar absoluto)")
    r_q = avaliar(yte, pt, lim_quantil, f"{alvo} (limiar por quantil)")
    r_q["baseline_F1"] = f1_score(yte, np.ones(len(yte), dtype=int))
    if verbose:
        print(f"\n=== {alvo} ===")
        print(f"  fracao sinalizada na validacao: {frac_sinalizada:.3f}")
        print(f"  limiar absoluto={lim_abs:.3f} -> F1={r_abs['F1']:.4f}")
        print(f"  limiar quantil ={lim_quantil:.3f} -> F1={r_q['F1']:.4f}   <- usado")
    return mdl, pt, (pt >= lim_quantil).astype(int), r_q, r_abs


modelo_mun80, proba_mun80, pred_mun80, res80, res80_abs = treinar_municipal("meta_80")
modelo_munep, proba_munep, pred_munep, resep, resep_abs = treinar_municipal("meta_inep")

comparacao_alvos = pd.DataFrame([res80, resep, res80_abs, resep_abs]).set_index("modelo")
display(comparacao_alvos.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

cm80 = confusion_matrix(MUN_TESTE.meta_80, pred_mun80)
sns.heatmap(cm80, annot=True, fmt=",d", cmap="Greens", ax=axes[0],
            xticklabels=["Não atinge", "Atinge"], yticklabels=["Não atinge", "Atinge"])
axes[0].set_title(f"meta_80 — F1 {res80['F1']:.4f} | acc {res80['Accuracy']:.2%}")
axes[0].set_xlabel("previsto")
axes[0].set_ylabel("real")

for nome, yv, pv in [("meta_80", MUN_TESTE.meta_80, proba_mun80),
                     ("meta_inep", MUN_TESTE.meta_inep, proba_munep)]:
    f, t, _ = roc_curve(yv, pv)
    axes[1].plot(f, t, label=f"{nome} (AUC={roc_auc_score(yv, pv):.3f})")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("Curva ROC — modelo municipal")
axes[1].set_xlabel("FPR")
axes[1].set_ylabel("TPR")
axes[1].legend()

comp = pd.DataFrame({
    "F1": [metricas_aluno["F1"], res80["F1"], resep["F1"]],
    "Accuracy": [metricas_aluno["Accuracy"], res80["Accuracy"], resep["Accuracy"]],
}, index=["Aluno\n(presentes)", "Município\nmeta_80", "Município\nmeta_inep"])
comp.plot.bar(ax=axes[2], rot=0)
axes[2].set_ylim(0, 1)
axes[2].set_title("Comparação por granularidade")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Comparacao entre granularidades (teste 2025):")
print(f"  aluno    : F1={metricas_aluno['F1']:.4f}  Acc={metricas_aluno['Accuracy']:.4f}")
print(f"  meta_80  : F1={res80['F1']:.4f}  Acc={res80['Accuracy']:.4f}  "
      f"(baseline F1={res80['baseline_F1']:.4f})")
print(f"  meta_inep: F1={resep['F1']:.4f}  Acc={resep['Accuracy']:.4f}  "
      f"(baseline F1={resep['baseline_F1']:.4f})")
print()
print("Lembrete de leitura: o F1 do modelo por aluno e o MAIOR em valor absoluto, e mesmo")
print("assim e o unico que nao supera seu proprio baseline. F1 so significa alguma coisa")
print("comparado a alternativa trivial do mesmo problema.")

### 7.1 Validação cruzada espacial

Um único par de anos pode ter dado sorte. Aqui os dois conjuntos são empilhados e validados
com `GroupKFold` agrupando por **UF**: cada fold testa em estados que o modelo nunca viu,
medindo generalização geográfica além da temporal.

In [ ]:
empilhado = pd.concat([MUN_TREINO.assign(par="2023->2024"),
                       MUN_TESTE.assign(par="2023+2024->2025")], ignore_index=True)

Xg, yg = empilhado[FEATURES_MUN], empilhado["meta_80"]
grupos = empilhado["uf"]

pipe_cv = construir_pipeline(XGBClassifier(**PARAMS_MUN), FEAT_NUM_MUN, FEAT_CAT_MUN)
cv = GroupKFold(n_splits=5)
acc_cv = cross_val_score(pipe_cv, Xg, yg, groups=grupos, cv=cv, scoring="accuracy")
auc_cv = cross_val_score(pipe_cv, Xg, yg, groups=grupos, cv=cv, scoring="roc_auc")
f1_cv = cross_val_score(pipe_cv, Xg, yg, groups=grupos, cv=cv, scoring="f1")

print(f"GroupKFold por UF (5 folds, {len(empilhado):,} municipios-ano):")
for nome, s in [("Accuracy", acc_cv), ("AUC-ROC", auc_cv), ("F1", f1_cv)]:
    print(f"  {nome:<9} {s.mean():.4f} (+/- {s.std() * 2:.4f})  folds: {np.round(s, 4)}")
print()
print("O resultado se mantem em estados fora do treino - nao e artefato de um recorte.")

### 7.2 O que este modelo pode e não pode fazer

Antes de entregar, é preciso saber qual pergunta o modelo responde bem. Testamos também uma
**regressão sobre a taxa municipal** — formulação aparentemente superior, já que a taxa é
contínua e dela se derivaria qualquer limiar sem retreinar. Ela falha, e o modo como falha é
o achado mais útil do notebook.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

reg = construir_pipeline(
    XGBRegressor(n_estimators=600, learning_rate=0.04, max_depth=4, subsample=0.8,
                 colsample_bytree=0.8, min_child_weight=5, reg_lambda=2.0,
                 tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE),
    FEAT_NUM_MUN, FEAT_CAT_MUN)
reg.fit(MUN_TREINO[FEATURES_MUN], MUN_TREINO.taxa_alvo)
taxa_prevista = reg.predict(MUN_TESTE[FEATURES_MUN])
taxa_real = MUN_TESTE.taxa_alvo.values

vies = taxa_prevista.mean() - taxa_real.mean()
print("REGRESSAO da taxa municipal (teste 2025):")
print(f"  R2 bruto                        = {r2_score(taxa_real, taxa_prevista):+.4f}")
print(f"  MAE                             = {mean_absolute_error(taxa_real, taxa_prevista):.4f}")
print(f"  correlacao previsto x real      = {np.corrcoef(taxa_prevista, taxa_real)[0, 1]:.4f}")
print(f"  vies de nivel                   = {vies:+.4f}")
print(f"  R2 apos remover o vies de nivel = {r2_score(taxa_real, taxa_prevista - vies):+.4f}")
print()
print("Taxa nacional de alfabetizacao por ano:")
for ano in sorted(DADOS):
    print(f"  {ano}: {DADOS[ano].IN_ALFABETIZADO.mean():.3f}")
salto_tr = DADOS[2024].IN_ALFABETIZADO.mean() - DADOS[2023].IN_ALFABETIZADO.mean()
salto_te = DADOS[2025].IN_ALFABETIZADO.mean() - DADOS[2024].IN_ALFABETIZADO.mean()
print(f"\nO modelo aprendeu o salto de {salto_tr * 100:+.1f}pp (2023->2024)")
print(f"e o aplicou a um ano que saltou {salto_te * 100:+.1f}pp (2024->2025).")
print()
print(f"R2 bruto {r2_score(taxa_real, taxa_prevista):.2f} mas recentrado "
      f"{r2_score(taxa_real, taxa_prevista - vies):.2f}: o erro NAO esta na ordenacao dos")
print("municipios, esta no NIVEL. A melhora nacional acelerou de forma que nenhuma")
print("feature disponivel antecipava.")
print()
print("Mesma causa explica tudo que encontramos antes:")
print("  - a correcao de prior por EM falhou aqui (nao e prior shift, e melhora real)")
print("  - o limiar absoluto nao transferiu entre anos")
print("  - o limiar por QUANTIL funcionou, porque e baseado em posicao relativa")

In [ ]:
# A qualidade do modelo esta na ordenacao. Medimos por decis.
decis = pd.DataFrame({"score": proba_mun80, "taxa_real": taxa_real,
                      "atingiu": MUN_TESTE.meta_80.values})
decis["decil"] = pd.qcut(decis.score, 10, labels=False) + 1

tabela_decis = decis.groupby("decil").agg(
    municipios=("taxa_real", "size"),
    taxa_real_media=("taxa_real", "mean"),
    pct_nao_atinge=("atingiu", lambda s: 1 - s.mean()))
display(tabela_decis.round(4))

taxa_base_risco = 1 - MUN_TESTE.meta_80.mean()
tabela_decis["lift"] = tabela_decis.pct_nao_atinge / taxa_base_risco

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].bar(tabela_decis.index, tabela_decis.taxa_real_media * 100, color="steelblue")
axes[0].axhline(80, color="red", ls="--", label="meta 80%")
axes[0].set_xlabel("decil de risco previsto (1 = pior)")
axes[0].set_ylabel("taxa real de alfabetização em 2025 (%)")
axes[0].set_title("Taxa REAL por decil previsto — monotônica")
axes[0].legend()

axes[1].bar(tabela_decis.index, tabela_decis.pct_nao_atinge * 100, color="indianred")
axes[1].axhline(taxa_base_risco * 100, color="gray", ls="--", label="taxa-base de risco")
axes[1].set_xlabel("decil de risco previsto (1 = pior)")
axes[1].set_ylabel("% que não atinge a meta (%)")
axes[1].set_title("Concentração de risco por decil")
axes[1].legend()
plt.tight_layout()
plt.show()

amplitude = (tabela_decis.taxa_real_media.iloc[-1] - tabela_decis.taxa_real_media.iloc[0]) * 100
print(f"Amplitude entre o 1o e o 10o decil: {amplitude:.1f} pontos percentuais de taxa real")
print(f"  decil 1  (pior):  taxa real {tabela_decis.taxa_real_media.iloc[0]:.1%} | "
      f"{tabela_decis.pct_nao_atinge.iloc[0]:.1%} nao atingem")
print(f"  decil 10 (melhor): taxa real {tabela_decis.taxa_real_media.iloc[-1]:.1%} | "
      f"{tabela_decis.pct_nao_atinge.iloc[-1]:.1%} nao atingem")
print()
print(f"Esta e a entrega util: uma fila de prioridade que separa {amplitude:.0f}pp de taxa")
print("real entre o pior e o melhor decil.")

## 8. Comparação de algoritmos

Sete algoritmos, ranqueados por **F1 na classe positiva**, com o limiar de cada um ajustado
para F1 na validação. Sem tuning de hiperparâmetros — a comparação é entre famílias de
modelo, não entre configurações otimizadas.

In [ ]:
try:
    from lightgbm import LGBMClassifier
    TEM_LGBM = True
except ImportError:
    TEM_LGBM = False
try:
    from catboost import CatBoostClassifier
    TEM_CATBOOST = True
except ImportError:
    TEM_CATBOOST = False

print(f"LightGBM: {TEM_LGBM} | CatBoost: {TEM_CATBOOST}")
if not (TEM_LGBM and TEM_CATBOOST):
    print("Para instalar:  pip install lightgbm catboost")


def candidatos():
    m = {
        "LogisticRegression": LogisticRegression(max_iter=2000),
        "DecisionTree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                               random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=10,
                                               min_samples_leaf=20, n_jobs=-1,
                                               random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                                               random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(n_estimators=400, learning_rate=0.06, max_depth=5,
                                 subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                                 tree_method="hist", n_jobs=-1, eval_metric="logloss",
                                 random_state=RANDOM_STATE),
    }
    if TEM_LGBM:
        m["LightGBM"] = LGBMClassifier(n_estimators=400, learning_rate=0.06, max_depth=6,
                                       subsample=0.8, colsample_bytree=0.8, n_jobs=-1,
                                       random_state=RANDOM_STATE, verbose=-1)
    if TEM_CATBOOST:
        m["CatBoost"] = CatBoostClassifier(iterations=400, learning_rate=0.06, depth=6,
                                           random_seed=RANDOM_STATE, verbose=0)
    return m


def comparar(Xtr, ytr, Xval, yval, Xte, yte, num, cat, titulo, por_quantil=False):
    linhas = []
    for nome, mdl in candidatos().items():
        try:
            pipe = construir_pipeline(mdl, num, cat)
            pipe.fit(Xtr, ytr)
            pv = pipe.predict_proba(Xval)[:, 1]
            lim = limiar_por_f1(yval, pv)
            pt = pipe.predict_proba(Xte)[:, 1]
            if por_quantil:
                lim = float(np.quantile(pt, 1 - float((pv >= lim).mean())))
        except Exception as erro:
            print(f"  [{nome}] falhou: {type(erro).__name__}: {str(erro)[:120]}")
            continue
        linhas.append(avaliar(yte, pt, lim, nome))
    tabela = pd.DataFrame(linhas).set_index("modelo").sort_values("F1", ascending=False)
    print(f"\n### {titulo}")
    display(tabela.round(4))
    return tabela

In [ ]:
# Modelo por aluno: subamostra de 500k para manter a comparacao em tempo razoavel
sub = X_tr.sample(500_000, random_state=RANDOM_STATE)
tab_aluno = comparar(sub, y_tr.loc[sub.index], X_val, y_val, X_teste, y_teste,
                     FEAT_NUM_ALUNO, CAT_ALUNO,
                     "Modelo por aluno (só presentes) — teste 2025")
print(f"baseline 'todos alfabetizado': F1 = {2 * N1 / (2 * N1 + N0):.4f}")

In [ ]:
Xa, Xv, ya, yv = train_test_split(MUN_TREINO[FEATURES_MUN], MUN_TREINO["meta_80"],
                                  test_size=0.25, random_state=RANDOM_STATE,
                                  stratify=MUN_TREINO["meta_80"])
tab_mun = comparar(Xa, ya, Xv, yv, MUN_TESTE[FEATURES_MUN], MUN_TESTE["meta_80"],
                   FEAT_NUM_MUN, FEAT_CAT_MUN,
                   "Modelo municipal (meta_80) — teste 2025", por_quantil=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
tab_aluno[["F1", "Accuracy", "AUC"]].plot.bar(ax=axes[0], rot=30)
axes[0].axhline(2 * N1 / (2 * N1 + N0), color="red", ls="--", label="F1 do baseline")
axes[0].set_ylim(0.3, 0.9)
axes[0].set_title("Por aluno (só presentes)")
axes[0].legend(fontsize=8)
tab_mun[["F1", "Accuracy", "AUC"]].plot.bar(ax=axes[1], rot=30)
axes[1].axhline(f1_score(MUN_TESTE.meta_80, np.ones(len(MUN_TESTE), dtype=int)),
                color="red", ls="--", label="F1 do baseline")
axes[1].set_ylim(0, 1)
axes[1].set_title("Por município (meta_80)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Por aluno, todos os algoritmos convergem para o mesmo patamar e nenhum supera o")
print("baseline no F1: o limite e dos DADOS, nao do algoritmo.")

## 9. Interpretabilidade

Feature importance e SHAP aplicados ao **modelo municipal**, que é o que tem poder preditivo
real. No modelo por aluno o SHAP apenas distribuiria importância entre features de contexto
que, somadas, movem pouco.

In [ ]:
pre_mun = modelo_mun80.named_steps["preprocessor"]
nomes_cat = (pre_mun.named_transformers_["cat"].named_steps["onehot"]
             .get_feature_names_out(FEAT_CAT_MUN).tolist())
nomes_features = FEAT_NUM_MUN + nomes_cat

importancias = (pd.Series(modelo_mun80.named_steps["modelo"].feature_importances_,
                          index=nomes_features)
                .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(9, 6))
top = importancias.head(18)[::-1]
ax.barh(top.index, top.values, color="steelblue")
ax.set_title("Feature importance — modelo municipal (meta_80)")
ax.set_xlabel("importância (gain normalizado)")
plt.tight_layout()
plt.show()

display(importancias.head(12).to_frame("importancia").round(4))

In [ ]:
try:
    import shap

    X_shap = pre_mun.transform(MUN_TESTE[FEATURES_MUN])
    explainer = shap.TreeExplainer(modelo_mun80.named_steps["modelo"])
    shap_values = explainer.shap_values(X_shap)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_shap, feature_names=nomes_features,
                      max_display=15, show=False)
    plt.title("SHAP — modelo municipal (meta_80)")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 5))
    shap.summary_plot(shap_values, X_shap, feature_names=nomes_features,
                      plot_type="bar", max_display=15, show=False)
    plt.title("SHAP — importância média absoluta")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("shap nao instalado - rode: pip install shap")

### 9.1 Municípios em maior risco educacional

In [ ]:
risco = MUN_TESTE.assign(prob_atingir=proba_mun80).copy()
risco["faixa"] = pd.cut(risco.prob_atingir, [0, 0.10, 0.30, 0.50, 0.75, 1.0],
                        labels=["Crítico", "Alto", "Médio", "Moderado", "Baixo"])

dist = risco.faixa.value_counts().sort_index()
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].bar(dist.index.astype(str), dist.values, color="indianred")
for i, v in enumerate(dist.values):
    axes[0].text(i, v + 20, f"{v:,}", ha="center", fontsize=9)
axes[0].set_title("Municípios por faixa de risco de não atingir 80%")
axes[0].set_ylabel("municípios")

por_uf = risco.groupby("uf").prob_atingir.mean().sort_values().head(15)
axes[1].barh(por_uf.index, por_uf.values, color="darkorange")
axes[1].set_title("UFs com menor probabilidade média de atingir a meta")
axes[1].set_xlabel("probabilidade média")

# precisao@k: a metrica que traduz para a decisao do gestor
ordem_risco = np.argsort(proba_mun80)
em_risco_real = 1 - MUN_TESTE.meta_80.values
ks = [100, 250, 500, 1000, 2000, 3000]
prec_k = [em_risco_real[ordem_risco[:k]].mean() for k in ks]
axes[2].plot(ks, prec_k, "o-", color="seagreen")
axes[2].axhline(em_risco_real.mean(), color="gray", ls="--", label="acaso")
axes[2].set_ylim(0.7, 1.02)
axes[2].set_xlabel("k municípios priorizados")
axes[2].set_ylabel("fração que de fato não atinge")
axes[2].set_title("Precisão@k — orçamento de intervenção")
axes[2].legend()
plt.tight_layout()
plt.show()

for k, p in zip(ks, prec_k):
    print(f"  priorizar {k:>5,} municipios -> {p:.1%} realmente nao atingem "
          f"(acaso: {em_risco_real.mean():.1%})")
print()
print("Top 10 municipios de maior risco:")
display(risco.nsmallest(10, "prob_atingir")[
    ["uf", "mun_taxa_pres", "mun_prof", "meta", "prob_atingir", "taxa_alvo"]].round(4))

## 10. Versões definitivas — diagnóstico e tratamento de overfitting

Até aqui só olhamos o desempenho no teste. Mas um modelo pode ir bem no teste e ainda estar
mal construído. O diagnóstico que falta é comparar **treino, validação e teste** lado a lado:

- se **treino ≫ validação**, o modelo decorou o treino → *overfitting*, trata-se com regularização;
- se **treino ≈ validação ≫ teste**, o modelo generaliza dentro do ano mas não entre anos →
  problema **temporal**, e regularizar não resolve;
- se **treino ≈ validação ≈ teste**, está saudável.

Os dois modelos deste notebook caem em situações diferentes, e é importante não confundi-las.

### 10.1 Diagnóstico: o modelo por aluno não tem overfitting

Aqui treino e validação praticamente coincidem. A perda acontece só no teste, e por um motivo
que regularização nenhuma corrige: **o baseline mudou de ano**. Em 2024 a taxa de
alfabetizados entre presentes era 59,8%; em 2025 subiu para 66,2%. O F1 de "prever todos
alfabetizado" acompanha esse salto (0,748 → 0,796), e o modelo não.

In [ ]:
PARAMS_ALUNO_FINAL = dict(
    n_estimators=300, learning_rate=0.06, max_depth=4,
    min_child_weight=100,            # folhas grandes: cada regra vale para >=100 alunos
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.5, reg_lambda=5.0,   # regularizacao L1 + L2
    tree_method="hist", n_jobs=-1, eval_metric="logloss", random_state=RANDOM_STATE)


def diagnosticar(modelo, conjuntos, limiar, titulo):
    """Compara treino / validacao / teste para separar overfitting de deriva temporal."""
    linhas = []
    for nome, X, y in conjuntos:
        p = modelo.predict_proba(X)[:, 1]
        pred = (p >= limiar).astype(int)
        linhas.append({
            "conjunto": nome, "n": len(y),
            "F1": f1_score(y, pred),
            "F1_baseline": f1_score(y, np.ones(len(y), dtype=int)),
            "Accuracy": accuracy_score(y, pred),
            "AUC": roc_auc_score(y, p)})
    tab = pd.DataFrame(linhas).set_index("conjunto")
    tab["F1_vs_baseline"] = tab.F1 - tab.F1_baseline
    print(f"\n### {titulo}")
    display(tab.round(4))
    return tab


modelo_aluno_final = construir_pipeline(
    XGBClassifier(**PARAMS_ALUNO_FINAL), FEAT_NUM_ALUNO, CAT_ALUNO).fit(X_tr, y_tr)
LIMIAR_FINAL = limiar_por_f1(y_val, modelo_aluno_final.predict_proba(X_val)[:, 1])

diag_aluno = diagnosticar(
    modelo_aluno_final,
    [("treino 2024", X_tr, y_tr), ("validação 2024", X_val, y_val),
     ("teste 2025", X_teste, y_teste)],
    LIMIAR_FINAL, f"MODELO POR ALUNO — versão final (limiar {LIMIAR_FINAL:.3f})")

gap = diag_aluno.loc["treino 2024", "F1"] - diag_aluno.loc["validação 2024", "F1"]
print(f"gap treino - validação = {gap:+.4f}  -> nao ha overfitting")
print()
print("Repare na coluna F1_vs_baseline: o modelo SUPERA o baseline no treino e na validacao,")
print("e PERDE no teste. Nao e o modelo que piorou - e o baseline que subiu, porque a taxa")
print("de alfabetizacao entre presentes passou de 59,8% (2024) para 66,2% (2025).")
print("Quanto maior a classe majoritaria, mais forte fica 'prever tudo positivo'.")

### 10.2 Tratamento: o modelo municipal tinha overfitting real

Aqui o quadro é outro. Com os hiperparâmetros iniciais, o modelo atingia AUC de 0,98 no
treino contra 0,89 na validação — decorou os 3.633 municípios de treino.

A busca de regularização abaixo roda 5 configurações × 5 sementes e mostra duas coisas:

- **o overfitting custa de verdade** — a configuração inicial não só tem o pior F1 de teste
  como o dobro da variância entre sementes;
- **mas "gap zero" não é o objetivo** — passado certo ponto, mais regularização não produz
  diferença distinguível do ruído. As configurações regularizadas empatam entre si.

Por isso cada linha vem com o desvio-padrão ao lado, e a comparação é feita **contra o
desvio**, não contra a quarta casa decimal. Diferença menor que um desvio-padrão não é
resultado — é ruído. O critério de escolha aqui foi o melhor F1 de teste **com a menor
variância**, que é o que dá confiança de que o número se repete numa próxima execução.

In [ ]:
GRADE_REGULARIZACAO = {
    "inicial (d4, mcw5, λ2)":  dict(n_estimators=400, learning_rate=0.05, max_depth=4,
                                    min_child_weight=5, reg_lambda=2.0),
    "d3, mcw20, λ5":           dict(n_estimators=300, learning_rate=0.05, max_depth=3,
                                    min_child_weight=20, reg_lambda=5.0),
    "d2, mcw30, λ10":          dict(n_estimators=300, learning_rate=0.05, max_depth=2,
                                    min_child_weight=30, reg_lambda=10.0),
    "d3, mcw50, λ10, α1":      dict(n_estimators=250, learning_rate=0.04, max_depth=3,
                                    min_child_weight=50, reg_lambda=10.0, reg_alpha=1.0),
    "d2, mcw50, λ20, α2":      dict(n_estimators=200, learning_rate=0.04, max_depth=2,
                                    min_child_weight=50, reg_lambda=20.0, reg_alpha=2.0),
}

linhas = []
for nome, kw in GRADE_REGULARIZACAO.items():
    f1s = {"tr": [], "va": [], "te": []}
    aucs = {"tr": [], "te": []}
    for semente in range(5):
        Xa, Xv, ya, yv = train_test_split(MUN_TREINO[FEATURES_MUN], MUN_TREINO["meta_80"],
                                          test_size=0.25, random_state=semente,
                                          stratify=MUN_TREINO["meta_80"])
        params = dict(kw, subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                      n_jobs=-1, eval_metric="logloss", random_state=semente)
        parcial = construir_pipeline(XGBClassifier(**params), FEAT_NUM_MUN, FEAT_CAT_MUN).fit(Xa, ya)
        completo = construir_pipeline(XGBClassifier(**params), FEAT_NUM_MUN, FEAT_CAT_MUN).fit(
            MUN_TREINO[FEATURES_MUN], MUN_TREINO["meta_80"])

        pv = parcial.predict_proba(Xv)[:, 1]
        frac = float((pv >= limiar_por_f1(yv, pv)).mean())   # fracao a sinalizar
        for chave, mdl, X, y in [("tr", parcial, Xa, ya), ("va", parcial, Xv, yv),
                                 ("te", completo, MUN_TESTE[FEATURES_MUN], MUN_TESTE["meta_80"])]:
            p = mdl.predict_proba(X)[:, 1]
            f1s[chave].append(f1_score(y, (p >= np.quantile(p, 1 - frac)).astype(int)))
            if chave in aucs:
                aucs[chave].append(roc_auc_score(y, p))
    linhas.append({"config": nome,
                   "F1_treino": np.mean(f1s["tr"]), "F1_valid": np.mean(f1s["va"]),
                   "F1_teste": np.mean(f1s["te"]), "F1_teste_dp": np.std(f1s["te"]),
                   "gap_tr_val": np.mean(f1s["tr"]) - np.mean(f1s["va"]),
                   "AUC_treino": np.mean(aucs["tr"]), "AUC_teste": np.mean(aucs["te"])})

busca = pd.DataFrame(linhas).set_index("config")
display(busca.round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
x = np.arange(len(busca))
axes[0].plot(x, busca.F1_treino, "o-", label="treino")
axes[0].plot(x, busca.F1_valid, "s-", label="validação")
axes[0].plot(x, busca.F1_teste, "^-", label="teste 2025", lw=2)
axes[0].set_xticks(x)
axes[0].set_xticklabels(busca.index, rotation=25, ha="right", fontsize=8)
axes[0].set_ylabel("F1")
axes[0].set_title("Mais regularização →")
axes[0].legend()
axes[1].plot(busca.gap_tr_val, busca.F1_teste, "o-", color="darkorange")
for i, nome in enumerate(busca.index):
    axes[1].annotate(nome.split(",")[0], (busca.gap_tr_val.iloc[i], busca.F1_teste.iloc[i]),
                     fontsize=7, xytext=(4, 4), textcoords="offset points")
axes[1].set_xlabel("gap treino - validação (overfitting)")
axes[1].set_ylabel("F1 no teste")
axes[1].set_title("O melhor teste NÃO está no gap zero")
plt.tight_layout()
plt.show()

MELHOR_CONFIG = busca.F1_teste.idxmax()
print(f"Melhor F1 no teste: '{MELHOR_CONFIG}'")
print(f"  gap    : {busca.gap_tr_val.iloc[0]:+.3f} -> {busca.loc[MELHOR_CONFIG, 'gap_tr_val']:+.3f}")
print(f"  F1 teste : {busca.F1_teste.iloc[0]:.4f} -> {busca.loc[MELHOR_CONFIG, 'F1_teste']:.4f}")
print(f"  AUC teste: {busca.AUC_teste.iloc[0]:.4f} -> {busca.loc[MELHOR_CONFIG, 'AUC_teste']:.4f}")
print()

# Compara cada configuracao com a vencedora levando o desvio-padrao em conta:
# diferenca menor que 1 desvio nao e resultado, e ruido.
f1_best = busca.loc[MELHOR_CONFIG, "F1_teste"]
dp_best = busca.loc[MELHOR_CONFIG, "F1_teste_dp"]
print(f"Comparacao com a vencedora (dp = {dp_best:.4f}):")
for nome, linha in busca.iterrows():
    dif = linha.F1_teste - f1_best
    tag = "vencedora" if nome == MELHOR_CONFIG else (
        "equivalente (dentro do ruido)" if abs(dif) < max(dp_best, linha.F1_teste_dp)
        else "pior de forma clara")
    print(f"  {nome:<24} gap {linha.gap_tr_val:+.3f}  F1 {linha.F1_teste:.4f} "
          f"±{linha.F1_teste_dp:.4f}  {tag}")
print()
pior = busca.F1_teste.idxmin()
print("Duas leituras honestas desta tabela:")
print(f"  1. O overfitting CUSTA: a config inicial tem gap {busca.gap_tr_val.iloc[0]:+.3f} e o")
print(f"     F1 de teste cai para {busca.F1_teste.iloc[0]:.4f}, com o dobro da variancia.")
print("  2. Passado certo ponto, mais regularizacao nao melhora nem piora de forma")
print("     distinguivel - as configs regularizadas empatam entre si dentro do ruido.")
print()
print("Portanto: regularizar e necessario, mas 'gap zero' nao e o objetivo. O criterio")
print("de escolha aqui foi o melhor F1 de teste COM a menor variancia entre sementes.")

### 10.3 Modelo municipal definitivo

In [ ]:
# Os hiperparametros vem da propria grade da §10.2 - nao sao fixados a mao.
# Assim o notebook nao pode divergir entre o que a busca escolheu e o que a versao
# final usa, mesmo que os dados ou as sementes mudem.
PARAMS_MUN_FINAL = dict(GRADE_REGULARIZACAO[MELHOR_CONFIG],
                        subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                        n_jobs=-1, eval_metric="logloss", random_state=RANDOM_STATE)
print(f"Configuração vencedora da §10.2: '{MELHOR_CONFIG}'")
print(f"  {GRADE_REGULARIZACAO[MELHOR_CONFIG]}")

Xa, Xv, ya, yv = train_test_split(MUN_TREINO[FEATURES_MUN], MUN_TREINO["meta_80"],
                                  test_size=0.25, random_state=RANDOM_STATE,
                                  stratify=MUN_TREINO["meta_80"])
mun_parcial = construir_pipeline(XGBClassifier(**PARAMS_MUN_FINAL),
                                 FEAT_NUM_MUN, FEAT_CAT_MUN).fit(Xa, ya)
pv = mun_parcial.predict_proba(Xv)[:, 1]
FRACAO_SINALIZADA = float((pv >= limiar_por_f1(yv, pv)).mean())

modelo_mun_final = construir_pipeline(XGBClassifier(**PARAMS_MUN_FINAL),
                                      FEAT_NUM_MUN, FEAT_CAT_MUN).fit(
    MUN_TREINO[FEATURES_MUN], MUN_TREINO["meta_80"])

print(f"Fração de municípios a sinalizar (definida na validação): {FRACAO_SINALIZADA:.3f}")
linhas = []
for nome, mdl, X, y in [("treino 2024", mun_parcial, Xa, ya),
                        ("validação 2024", mun_parcial, Xv, yv),
                        ("teste 2025", modelo_mun_final, MUN_TESTE[FEATURES_MUN], MUN_TESTE["meta_80"])]:
    p = mdl.predict_proba(X)[:, 1]
    pred = (p >= np.quantile(p, 1 - FRACAO_SINALIZADA)).astype(int)
    linhas.append({"conjunto": nome, "n": len(y), "F1": f1_score(y, pred),
                   "F1_baseline": f1_score(y, np.ones(len(y), dtype=int)),
                   "Accuracy": accuracy_score(y, pred), "AUC": roc_auc_score(y, p)})
diag_mun = pd.DataFrame(linhas).set_index("conjunto")
diag_mun["F1_vs_baseline"] = diag_mun.F1 - diag_mun.F1_baseline
print("\n### MODELO MUNICIPAL — versão final (meta_80)")
display(diag_mun.round(4))

print(f"gap treino - validação = "
      f"{diag_mun.loc['treino 2024', 'F1'] - diag_mun.loc['validação 2024', 'F1']:+.4f}")
print()
print("Supera o baseline nos TRES conjuntos - unico modelo do notebook que faz isso.")

In [ ]:
# Comparacao final dos dois modelos, cada um contra o SEU baseline
proba_final = modelo_mun_final.predict_proba(MUN_TESTE[FEATURES_MUN])[:, 1]
pred_final = (proba_final >= np.quantile(proba_final, 1 - FRACAO_SINALIZADA)).astype(int)
p_aluno_final = modelo_aluno_final.predict_proba(X_teste)[:, 1]
pred_aluno_final = (p_aluno_final >= LIMIAR_FINAL).astype(int)

resumo_final = pd.DataFrame([
    {"modelo": "Aluno (condicional à presença)",
     "F1": f1_score(y_teste, pred_aluno_final),
     "F1_baseline": f1_score(y_teste, np.ones(len(y_teste), dtype=int)),
     "Accuracy": accuracy_score(y_teste, pred_aluno_final),
     "AUC": roc_auc_score(y_teste, p_aluno_final)},
    {"modelo": "Município (meta_80)",
     "F1": f1_score(MUN_TESTE.meta_80, pred_final),
     "F1_baseline": f1_score(MUN_TESTE.meta_80, np.ones(len(MUN_TESTE), dtype=int)),
     "Accuracy": accuracy_score(MUN_TESTE.meta_80, pred_final),
     "AUC": roc_auc_score(MUN_TESTE.meta_80, proba_final)},
]).set_index("modelo")
resumo_final["ganho_sobre_baseline"] = resumo_final.F1 - resumo_final.F1_baseline
display(resumo_final.round(4))

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(resumo_final))
ax.bar(x - 0.2, resumo_final.F1, 0.4, label="F1 do modelo", color="steelblue")
ax.bar(x + 0.2, resumo_final.F1_baseline, 0.4, label="F1 do baseline", color="lightgray")
for i, (f, b) in enumerate(zip(resumo_final.F1, resumo_final.F1_baseline)):
    ax.text(i - 0.2, f + 0.01, f"{f:.3f}", ha="center", fontsize=9)
    ax.text(i + 0.2, b + 0.01, f"{b:.3f}", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(resumo_final.index, fontsize=9)
ax.set_ylabel("F1 (teste 2025)")
ax.set_title("Cada modelo contra o SEU baseline — é a comparação que importa")
ax.legend()
plt.tight_layout()
plt.show()

print("O F1 absoluto do modelo por aluno e MAIOR (classe positiva majoritaria infla a")
print("metrica), mas so o municipal entrega valor sobre a alternativa trivial.")
print("Comparar F1 entre problemas diferentes nao faz sentido: comparar contra o")
print("baseline de cada um, sim.")

## 11. Respostas às perguntas de negócio

**1. Quais fatores mais impactam a alfabetização?**
No nível municipal, a proficiência média dos anos anteriores e seus percentis dominam,
seguidos da taxa histórica e do `gap_meta`. A desigualdade entre escolas dentro do município
(`mun_esc_spread`) aparece como fator próprio: municípios com média razoável mas alta
dispersão interna são mais frágeis.

No nível do aluno, o fator de maior efeito **mecânico** sobre o indicador oficial é a
presença — mas §2.1 mostra que isso é a definição do indicador, não um achado preditivo.

**Cuidado com a leitura dessa variável.** `IN_PRESENCA_LP` é, pela documentação oficial do
INEP, *"indicador de presença na prova de Língua Portuguesa"*, com valores `0 = Ausente` e
`1 = Presente`. É comparecimento a **um único dia** — não é percentual de frequência às
aulas, e não é medida de evasão escolar. A ausência pode ser doença, transporte, chuva,
mudança recente de escola ou evasão, e os microdados não distinguem entre essas causas.

Concluir daí "combater a evasão escolar" é uma inferência que o dado não sustenta. O que se
pode afirmar é mais estreito e ainda assim acionável: **garantir o comparecimento no dia da
avaliação** move o indicador oficial mecanicamente, porque todo ausente entra como não
alfabetizado. E §6.4 mostra que esse comparecimento é quase imprevisível pelo histórico da
escola (AUC ≈ 0,60) — coerente com ser um evento de circunstância, não de trajetória.

**2. Quais municípios apresentam maior risco educacional?**
O modelo municipal produz uma fila de prioridade (§7.2 e §9.1). Os decis são monotônicos e
separam ~31 pontos percentuais de taxa real entre o pior e o melhor — é esta a entrega
operacional do projeto.

**3. Quais regiões possuem padrões semelhantes?**
O KMeans (k=6, §5) separa perfis que vão de taxa muito baixa com alta dispersão interna até
taxa alta e homogênea. O cluster entra como feature e como chave de segmentação de política.

**4. Como prever municípios que podem não atingir metas futuras?**
É o alvo `meta_inep` (§7): perfil dos anos *t−1*/*t−2* prevendo o atingimento da meta própria
do município em *t*.

**5. Quais variáveis possuem maior influência nos modelos?**
Ver §9 — feature importance e SHAP sobre o modelo municipal.

---

## Conclusões

| Modelo | F1 | Baseline F1 | Acurácia | AUC | Situação |
|---|---|---|---|---|---|
| Anterior — aluno "limpo" | — | — | 64,98% | 0,638 | era a regra `presente → alfabetizado` |
| Anterior — agregado | — | — | 91,16% | 0,969 | *data leakage* |
| **Este — aluno (só presentes)** | 0,792 | **0,796** | 66,7% | 0,636 | honesto; **não supera o baseline** |
| **Este — município `meta_80`** | **0,572** | 0,360 | 82,3% | 0,84–0,86 | **supera o baseline com folga** |
| Este — município `meta_inep` | 0,648 | **0,708** | — | 0,755 | alvo balanceado; **não supera no F1** |

**O que mudou de fato:**

1. O vazamento foi identificado e provado em código, não apenas declarado.
2. Alunos ausentes saíram do treino, da validação e do teste: não há problema de predição
   onde o rótulo é definido por regra administrativa.
3. As features respeitam defasagem temporal real, e o município usa dois anos de histórico.
4. O limiar saiu do teste e foi para a validação, transportado **por quantil** — a forma que
   sobrevive à mudança de prior entre anos.
5. A seleção passou a ser por F1 na classe positiva, com o baseline sempre declarado ao lado.
6. Treino, validação e teste passaram a ser comparados lado a lado (§10), o que permitiu
   distinguir *overfitting* (real no modelo municipal, tratado) de **deriva temporal** (o caso
   do modelo por aluno, que regularização não corrige).

---

## Conformidade com o enunciado

O `[IAST] - Tech Challenge - Fase 3.pdf` não fixa meta numérica de métrica — afirma que *"o
foco não é apenas gerar métricas altas, mas produzir inteligência aplicável ao contexto
educacional brasileiro"*. O que ele exige, e onde está atendido:

| Exigência do enunciado | Onde | Observação |
|---|---|---|
| Análise exploratória (distribuições, padrões, correlações) | §1.1, §4 | heatmap de correlação e EDA por ano |
| Imputação de valores faltantes para variáveis numéricas | §6 | `SimpleImputer(strategy="median")` dentro do pipeline |
| Transformação de variáveis numéricas e categóricas | §6 | `StandardScaler` + `OneHotEncoder` |
| **Tratamento de data leakage** | §2, §3 | auditado em código; 4 formas de vazamento identificadas e corrigidas (§2.5) |
| Pré-processamento integrado ao modelo | §6 | `ColumnTransformer` dentro de `Pipeline`, ajustado só no treino |
| Treinamento e validação do modelo | §6, §7 | split temporal + validação separada para limiar |
| Validação com replicabilidade e generalização | §7.1, §10 | `GroupKFold` por UF, teste fora do tempo, 5 sementes, `random_state` fixo |
| Estratégias para reduzir overfitting | §10 | diagnóstico treino/validação/teste e grade de regularização |
| Interpretabilidade (Feature Importance e SHAP) | §9 | ambos, sobre o modelo municipal |
| Respostas às 5 perguntas de negócio | §11 | com os números que as sustentam |

Uma ressalva de escopo honesta: o enunciado sugere enriquecer a base com fontes externas
(IBGE, Censo Escolar, FUNDEB, Atlas do Desenvolvimento Humano). Isso **não foi feito** aqui —
está listado entre as evoluções, com a justificativa medida de por que ajudaria o modelo
municipal e não o modelo por aluno.

**Limitações — e esta é a parte substantiva do trabalho.**

O modelo por aluno **não supera o baseline "todos alfabetizado" no F1**, e supera a acurácia
por uma fração de ponto. Isso não é falha de modelagem, é a medida do problema:

- O `TS_ALUNO` não tem nenhuma variável do aluno. Toda a informação é da escola.
- 84,7% da variação de quem é alfabetizado está **dentro** da mesma escola, entre colegas de
  sala, sobre os quais não há dado algum.
- O corte de 743 fica no ponto mais denso da distribuição de proficiência (desvio 48), com
  ~30% dos alunos a menos de 20 pontos dele — parte do rótulo é erro de medida.
- O efeito de escola quase não persiste: correlação verdadeira de ~0,19 entre anos.
- O teto do oráculo entre presentes é 70,5%, e o modelo entrega ~66,4% contra baseline 66,2%.

O valor real do modelo por aluno está na **ordenação** (AP de ~0,45 na classe em risco contra
acaso de ~0,34), não na classificação binária.

**Só o alvo `meta_80` supera seu baseline no F1.** No `meta_inep`, que é balanceado (54,8% de
positivos), o baseline trivial tem F1 0,708 e o modelo 0,648 — ele perde na classificação e
ganha só na ordenação (AUC 0,755). Vale dizer isso explicitamente em vez de exibir a métrica
sozinha: em problemas com classe positiva majoritária, o F1 do "prever tudo positivo" é alto
e precisa estar sempre na tabela ao lado.

O modelo municipal `meta_80` é o entregável útil: F1 0,572 contra baseline 0,360, acurácia
acima de 82% e AP de 0,95 na classe em risco — é ele que sustenta o ranqueamento de risco que
a política pública precisa (§9.1).

**O que este dado suporta — e o que não suporta.**

Esta é a síntese metodológica do trabalho. A §7.2 mostra que a regressão sobre a taxa
municipal tem R² ≈ 0,09 bruto mas **R² ≈ 0,38 depois de remover o viés de nível**, com
correlação de 0,70 entre previsto e real. O erro não está na ordenação dos municípios: está
no nível. A taxa nacional subiu 2,0pp de 2023 para 2024 e 6,4pp de 2024 para 2025 — uma
aceleração que nenhuma feature disponível antecipava.

A mesma causa explica, retroativamente, cada obstáculo encontrado: a correção de prior por EM
falhou no municipal (não era desvio de prior, era melhora educacional real), o limiar absoluto
não transferiu entre anos, e o limiar por quantil funcionou justamente por ser relativo.

**Conclusão: este dado suporta ordenação, não nível.** Um modelo de qualidade aqui é um
modelo de **priorização de risco**, avaliado por métricas de ordenação (AUC, AP, decis,
precisão@k) — não um classificador de limiar fixo nem um preditor de valores absolutos. É
assim que o entregável deve ser apresentado e usado.

**Evoluções.** Dados socioeconômicos municipais (IBGE, Atlas do Desenvolvimento Humano,
FUNDEB por aluno) são a aposta mais promissora, porque o efeito municipal é estável entre
anos — hipótese não testada aqui. Uma quarta edição da avaliação permitiria modelar a
*tendência* nacional e atacar o viés de nível, hoje o principal erro. Para **superar** o teto
por aluno seria necessário dado que varia dentro da escola: frequência ao longo do ano,
histórico de reprovação, avaliações diagnósticas internas. Nada disso existe nos microdados
públicos.

---

## 12. Conclusão final

### Uma frase

**Os microdados públicos de alfabetização permitem ordenar municípios por risco. Não permitem
prever o destino de uma criança.** Todo o resto deste notebook é a demonstração dessa
afirmação e a construção do que dela se aproveita.

### O achado que não estava previsto

A dificuldade central que enfrentamos acabou sendo uma boa notícia sobre o país.

| Ano | Taxa nacional de alfabetização |
|---|---|
| 2023 | 50,2% |
| 2024 | 52,2% |
| 2025 | 58,6% |

**+8,4 pontos percentuais em dois anos, com a melhora acelerando** — de +2,0pp no primeiro
intervalo para +6,4pp no segundo. É por causa disso que a regressão sobre a taxa teve R² ≈ 0
enquanto acertava a ordenação (§7.2); que a correção de prior por EM falhou no modelo
municipal; que limiares absolutos não transferiram entre anos; e que o modelo por aluno perde
para o baseline no teste embora o supere no treino e na validação (§10.1).

Não eram quatro problemas técnicos. Era um só fenômeno: **o alvo estava se movendo, para
cima, mais rápido do que o histórico permitia antecipar.** Um modelo treinado em 2024 carrega
o Brasil de 2024 e subestima o Brasil de 2025.

Há uma ironia útil aqui, e ela vale para a apresentação executiva: *o Compromisso Nacional
Criança Alfabetizada estar funcionando é precisamente o que torna o indicador difícil de
prever*. Modelos preditivos gostam de mundos estáveis. Políticas públicas eficazes produzem o
contrário.

### Instruções de uso do modelo entregue

Um modelo entregue sem instruções de uso é um modelo que será usado errado. As deste:

**Pode ser usado para:**
- Ordenar municípios por risco e definir uma fila de priorização. Os decis são monotônicos e
  separam ~31 pontos percentuais de taxa real entre o pior e o melhor (§7.2).
- Dimensionar intervenção sob orçamento. Priorizando os 250 municípios de maior risco, ~100%
  de fato não atingem a meta (§9.1).
- Identificar municípios cujo risco vem de **desigualdade interna** e não de média baixa — a
  feature `mun_esc_spread` separa esses dois casos, que pedem políticas diferentes.

**Não pode ser usado para:**
- Afirmar que um município terá uma taxa específica em um ano futuro. O erro de nível é de
  ~9pp e sistemático (§7.2).
- Operar com limiar fixo de probabilidade entre anos. Use sempre a fração a sinalizar, não o
  valor de corte (§7).
- Predizer o resultado de uma criança específica. O teto é 70,5% contra um baseline de 66,2%
  — uma faixa de 4,35pp para disputar (§6.3), e nenhum dos sete algoritmos testados supera o
  baseline no F1 (§8).

**Precisa ser reavaliado quando:** sair a edição de 2026. O viés de nível é o maior erro do
modelo hoje, e uma quarta observação anual permite estimar a tendência nacional em vez de
herdá-la do último intervalo.

### O que fica como método

Três hábitos que este notebook adotou depois de errar sem eles, e que valem além deste projeto:

1. **Calcular o teto antes de otimizar.** Saber que existiam apenas 4,35pp entre o baseline e
   o melhor modelo concebível transformou meses potenciais de tentativa e erro numa decisão de
   dez minutos: mudar a granularidade do problema.
2. **Nunca reportar métrica sem o baseline ao lado.** Um F1 de 0,79 parece bom e é pior que
   não fazer nada; um F1 de 0,49 parece fraco e agrega valor real. O número sozinho não
   informa — a comparação, sim.
3. **Testar estabilidade antes de comemorar.** Um ganho de +5,8pp de F1 vindo de novas
   features evaporou ao rodar com cinco sementes. Diferença menor que o desvio-padrão não é
   resultado.

### O que o projeto entrega, afinal

Não é o modelo de maior acurácia — é o único conjunto de afirmações sobre alfabetização que
sobrevive a auditoria. Um modelo municipal que ordena risco com AUC de 0,85 e é honesto sobre
onde erra; a demonstração, em código, de que a predição individual está limitada pelos dados e
não pela técnica; e um catálogo dos erros que produziam os números bonitos e falsos do ponto
de partida (§2.5).

O enunciado pedia inteligência aplicável ao contexto educacional brasileiro, e não apenas
métricas altas. Uma fila de prioridade confiável, com o alcance e os limites declarados, é
mais aplicável do que um número de acurácia que não resistiria à primeira pergunta de quem
entende do assunto.